# Data Wrangling + In-Depth EDA — Masterclass

The first masterclass in the **AI/ML engineering data series**. The goal is not "learn pandas." The goal is: when an interviewer pushes a dataset in front of you, you have a **rehearsed inspection routine**, you classify every column correctly within 5 minutes, and you ask the questions that signal you've done this before.

## Why this notebook exists
EDA is the moment where interviewers learn three things about you at once:
1. **Do you know the tools** (NumPy, Pandas, basic viz)?
2. **Do you have intuition about data** (types, distributions, missingness, leakage)?
3. **Do you ask the right questions** (what is one row? what's the target? how was this sampled?)?

Most candidates score on (1), partially on (2), and zero on (3). This notebook trains all three.

## Stack (May 2026)
- Python 3.11+
- **pandas 3.0.x** — Copy-on-Write is now default, StringDtype is the default for strings
- **numpy 2.4.x** — modern `Generator` API via `np.random.default_rng`
- matplotlib + seaborn for plots
- scipy.stats for distribution tests
- scikit-learn for the embedding-cluster preview at the end

## How to read this notebook
**11 parts, ~140 cells.** Each section follows: short story → concept → why it matters → code → "what to say in the interview" callout.

You can read top-to-bottom, or jump. The single most valuable section for interviews is **Part 3 — The First 5 Minutes Framework**.


## Setup

In [ ]:
# Standard imports used throughout the notebook.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

# Reproducibility: NumPy's modern Generator API (do NOT use np.random.seed in 2026 code).
RNG = np.random.default_rng(seed=42)

# Display settings: show enough columns/rows without spamming.
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
pd.set_option("display.precision", 3)

# Seaborn theme — clean and consistent across all plots in this notebook.
sns.set_theme(style="whitegrid", context="notebook")

print(f"pandas {pd.__version__}  |  numpy {np.__version__}  |  seaborn {sns.__version__}")


### Why `np.random.default_rng` and not `np.random.seed`?

`np.random.seed` mutates a global state — it's the equivalent of a global variable. The modern `Generator` API is **stateless from your code's perspective**: you pass `RNG` around, and reproducibility is local to whoever holds the generator. Every NumPy 2.x docstring uses this pattern. If you write `np.random.rand()` in an interview in 2026, it's a tell that you learned NumPy from a 2019 tutorial.


---
# Part 1 — NumPy foundations for wrangling

You won't write much raw NumPy in an EDA interview — pandas wraps it. But you need to **know what's happening underneath**, because every pandas operation that's fast is fast because NumPy is doing it in C.

This part covers the NumPy concepts that bleed into pandas: dtypes, axes, broadcasting, vectorization, and indexing.

## 1.1 `ndarray` vs Python list — why NumPy

A Python list is a heterogeneous, pointer-based container. A NumPy `ndarray` is a **contiguous block of typed memory**. That single difference enables everything: vectorized math, broadcasting, and the speed.

In [ ]:
# Same data, two containers.
py_list = list(range(1_000_000))
np_arr  = np.arange(1_000_000)

# Memory layout (approximate):
# - Python list: a pointer array of 1M pointers, each pointing to a PyInt object on the heap.
# - NumPy array: 1M int64 values packed contiguously in one buffer.
# That's why NumPy is ~50x smaller and ~100x faster for math.

# Quick memory comparison (sys.getsizeof for the list is incomplete because it only counts the
# pointer array, not the boxed ints, but the order of magnitude is clear).
import sys
print(f"list size (pointers only): {sys.getsizeof(py_list):>12,} bytes")
print(f"ndarray size:              {np_arr.nbytes:>12,} bytes")


In [ ]:
# Vectorization benchmark: square every element.
import time

py_list = list(range(1_000_000))
np_arr  = np.arange(1_000_000)

t0 = time.perf_counter()
_ = [x * x for x in py_list]
t_py = time.perf_counter() - t0

t0 = time.perf_counter()
_ = np_arr ** 2
t_np = time.perf_counter() - t0

print(f"Python list comprehension: {t_py*1000:>7.1f} ms")
print(f"NumPy vectorized:          {t_np*1000:>7.1f} ms")
print(f"Speedup:                   {t_py/t_np:>7.1f}x")


**Interview line:** "NumPy is fast because the data is contiguous, typed, and the loop is in C. Pandas inherits this — every fast pandas op is delegating to NumPy or, in 3.0, to PyArrow."


## 1.2 dtype, shape, axes — the mental model

Every `ndarray` has three things you should be able to recite:
- **`dtype`** — the type of every element (`int64`, `float64`, `bool`, etc.)
- **`shape`** — the size along each dimension, e.g. `(rows, cols)`
- **`ndim`** — number of dimensions (1 for vector, 2 for matrix, N for tensor)

In [ ]:
# A 4x3 array of floats.
a = np.array([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
    [7.0, 8.0, 9.0],
    [10., 11., 12.],
])

print(f"dtype : {a.dtype}")
print(f"shape : {a.shape}       <- (rows=4, cols=3)")
print(f"ndim  : {a.ndim}")
print(f"size  : {a.size}       <- total number of elements")
print(f"nbytes: {a.nbytes}      <- bytes used in memory (12 floats * 8 bytes each)")


### The `axis=0` vs `axis=1` trap

This is the single most-asked NumPy/Pandas trick question. The rule:

- **`axis=0`** = collapse rows = operate **down columns** = return one value per column
- **`axis=1`** = collapse cols = operate **across rows** = return one value per row

Memory trick: "axis is the dimension you're **getting rid of**." If you sum along axis=0, you get rid of the row dimension and are left with one number per column.

In [ ]:
# Same 4x3 array.
print("Original array (shape 4x3):")
print(a)

print("\nSum along axis=0 (collapses rows → one value per column):")
print(a.sum(axis=0))   # shape (3,)

print("\nSum along axis=1 (collapses cols → one value per row):")
print(a.sum(axis=1))   # shape (4,)

# In pandas this exact rule applies:
#   df.sum(axis=0)  -> sum per column (the default)
#   df.sum(axis=1)  -> sum per row


## 1.3 Broadcasting — and the 3 bugs it causes

Broadcasting lets NumPy operate on arrays of different shapes by **virtually stretching** the smaller one. The rules:
1. Align shapes from the right.
2. Dimensions of size 1 stretch to match.
3. Missing dimensions are treated as size 1.

If neither array has size 1 in a mismatched dimension, you get an error.

In [ ]:
# Centering each column by its mean — the canonical broadcasting example.
X = RNG.normal(loc=[10, 100, 1000], scale=[1, 5, 50], size=(5, 3))
print("Data X (shape 5x3):")
print(X.round(2))

col_means = X.mean(axis=0)         # shape (3,)
print("\nColumn means (shape (3,)):", col_means.round(2))

# Broadcasting: (5,3) - (3,)  ->  (5,3) - (1,3) virtually  ->  (5,3)
X_centered = X - col_means
print("\nCentered X (column means are now ~0):")
print(X_centered.round(2))
print("\nNew column means (should be ~0):", X_centered.mean(axis=0).round(6))


In [ ]:
# The classic broadcasting bug: trying to center by row mean.
# We want X - row_means, but row_means has shape (5,), and (5,3) - (5,) FAILS.

row_means = X.mean(axis=1)
print("row_means shape:", row_means.shape, "  X shape:", X.shape)

try:
    X - row_means
except ValueError as e:
    print(f"\nFails with: {e}")

# The fix: reshape row_means to (5,1) so it broadcasts across columns.
X_row_centered = X - row_means.reshape(-1, 1)
# Equivalently: X - row_means[:, np.newaxis]
print("\nRow-centered shape:", X_row_centered.shape, "  row means now:", X_row_centered.mean(axis=1).round(6))


**Three bugs to expect:**
1. **Shape-mismatch ValueError** — the one above. Fix: `reshape(-1, 1)` or `arr[:, np.newaxis]`.
2. **Silent broadcasting** — `(3,)` broadcasts against `(1, 3)` cleanly, but maybe you wanted `(3, 1)`. The op succeeds but the result is wrong.
3. **In-place broadcasting** — `a += b` requires `b` to broadcast *into* `a`'s shape; if `a` is smaller, you get an error.


## 1.4 Indexing — basic, fancy, boolean

In [ ]:
# Set up a small array for clarity.
x = np.array([10, 20, 30, 40, 50, 60, 70, 80, 90])

# Basic slicing — returns a VIEW (shares memory with original).
print("x[2:5]      :", x[2:5])             # [30 40 50]
print("x[::2]      :", x[::2])             # [10 30 50 70 90]
print("x[::-1]     :", x[::-1])            # reverse

# Fancy indexing — pass an array of indices, returns a COPY.
idx = np.array([0, 3, 6])
print("x[[0,3,6]]  :", x[idx])             # [10 40 70]

# Boolean indexing — pass a boolean mask of same length, returns a COPY.
mask = x > 40
print("mask        :", mask)
print("x[x > 40]   :", x[mask])            # [50 60 70 80 90]

# Combine: which elements are even AND greater than 30?
combined = (x % 2 == 0) & (x > 30)         # use & not 'and' (element-wise)
print("combined    :", x[combined])


**Interview gotcha:** boolean masks must be combined with `&`, `|`, `~` (bitwise) — **not** `and`, `or`, `not` (Python keywords). Forgetting this is a tell. Also: parenthesize each condition or operator precedence will bite you.

## 1.5 Aggregations along axes

In [ ]:
# Realistic EDA aggregation: summary stats per column.
data = RNG.normal(loc=50, scale=10, size=(100, 4))

# All these accept axis= and return one value per column when axis=0.
print(f"Mean per column:    {data.mean(axis=0).round(2)}")
print(f"Std  per column:    {data.std(axis=0, ddof=1).round(2)}")  # ddof=1 -> sample std
print(f"Median per column:  {np.median(data, axis=0).round(2)}")
print(f"5/95 percentiles:   {np.percentile(data, [5, 95], axis=0).round(2)}")
print(f"Min:                {data.min(axis=0).round(2)}")
print(f"Max:                {data.max(axis=0).round(2)}")

# Interview note: ddof=1 (sample std) vs ddof=0 (population std). Pandas defaults to ddof=1
# but NumPy defaults to ddof=0. Mismatch shows up in interview pop quizzes.


## 1.6 `np.where`, `np.select`, `np.clip`, `np.percentile`

These four cover ~90% of the array-level transforms you'll do in EDA.

In [ ]:
x = np.array([-5, -1, 0, 3, 8, 15, 100])

# np.where(cond, if_true, if_false) — vectorized ternary.
sign = np.where(x >= 0, "non_neg", "neg")
print("sign:        ", sign)

# np.select — multiple conditions in order, like a switch.
conditions = [x < 0, x == 0, (x > 0) & (x <= 10), x > 10]
choices    = ["neg",  "zero",  "small_pos",         "big_pos"]
bucketed = np.select(conditions, choices, default="unknown")
print("bucketed:    ", bucketed)

# np.clip — cap values to a range. Universal outlier squashing tool.
clipped = np.clip(x, a_min=0, a_max=20)
print("clipped:     ", clipped)

# np.percentile (or np.quantile) — pandas .describe() lives on top of this.
p5, p50, p95 = np.percentile(x, [5, 50, 95])
print(f"p5/50/95:    {p5:.1f} / {p50:.1f} / {p95:.1f}")


## 1.7 NaN handling at the array level

In [ ]:
# NaN propagates through every NumPy op. This is by design — silent NaN would be worse.
arr = np.array([1.0, 2.0, np.nan, 4.0, 5.0])
print(f"arr.sum()    = {arr.sum()}        <- NaN propagates")
print(f"arr.mean()   = {arr.mean()}        <- still NaN")

# Use nan-aware functions to skip NaNs.
print(f"np.nansum()  = {np.nansum(arr)}        <- skips NaN")
print(f"np.nanmean() = {np.nanmean(arr)}")
print(f"np.nanstd()  = {np.nanstd(arr, ddof=1):.4f}")

# Detect NaN — DO NOT use ==, NaN is not equal to anything (including itself).
print(f"\nnp.nan == np.nan   -> {np.nan == np.nan}  (always False)")
print(f"np.isnan(arr)      -> {np.isnan(arr)}")
print(f"count of NaN       -> {np.isnan(arr).sum()}")


---
**Part 1 checklist** — can you, without looking:
- [ ] Explain why NumPy is faster than a list (contiguous typed memory + C loops)?
- [ ] State the `axis=0` vs `axis=1` rule (axis is the dimension you collapse)?
- [ ] Reshape a (5,) row-mean to broadcast against a (5,3) matrix?
- [ ] Combine boolean masks with `&`/`|`/`~` and parenthesize them?
- [ ] Know that NaN propagates and use `nansum`/`nanmean` to skip it?


---
# Part 2 — Pandas 3.0 foundations

Pandas 3.0 (Jan 2026) introduced two changes you must mention in an interview:
1. **Copy-on-Write (CoW) is now the default**. The legendary `SettingWithCopyWarning` is gone. Mutations via chained indexing now either work cleanly or raise — no silent corruption.
2. **`StringDtype` is the default for string columns**, not `object`. Strings now have proper dtype semantics, missing values as `pd.NA`, and PyArrow-backed memory.

If you say "I store strings as `object` and use `df.copy()` everywhere to avoid `SettingWithCopyWarning`", you sound like you stopped reading the changelog in 2023. Below we'll use the modern idioms throughout.

## 2.1 Series, DataFrame, Index — the three building blocks

In [ ]:
# Series: a labeled 1-D array. Has values + an Index.
s = pd.Series([10, 20, 30, 40], index=["a", "b", "c", "d"], name="sales")
print("Series:")
print(s)
print(f"\ns.values -> ndarray of shape {s.values.shape}")
print(f"s.index  -> {s.index}")
print(f"s.name   -> {s.name}")


In [ ]:
# DataFrame: a labeled 2-D table. It's a dict of Series sharing one Index.
df = pd.DataFrame({
    "age":      [25, 32, 47, 51, 23],
    "city":     ["Hyderabad", "Bangalore", "Mumbai", "Delhi", "Pune"],
    "salary":   [80_000, 120_000, 200_000, 250_000, 65_000],
    "joined":   pd.to_datetime(["2023-01-15", "2021-06-01", "2018-03-20", "2015-11-09", "2024-02-28"]),
})
print(df)
print(f"\ndtypes:\n{df.dtypes}")


**Notice the dtypes:** `age` → int64, `city` → `str` (pandas 3.0 default StringDtype), `salary` → int64, `joined` → datetime64[ns]. Pandas infers types on construction.

## 2.2 The dtype zoo that matters in interviews

In [ ]:
# Build a frame that has every dtype you'd see in a real dataset.
dtype_demo = pd.DataFrame({
    "int_col":      pd.array([1, 2, 3, 4], dtype="int64"),
    "int_nullable": pd.array([1, 2, None, 4], dtype="Int64"),   # capital I -> nullable
    "float_col":    pd.array([1.5, 2.5, np.nan, 4.5], dtype="float64"),
    "bool_col":     pd.array([True, False, True, False], dtype="bool"),
    "bool_nullable":pd.array([True, False, None, True], dtype="boolean"),
    "string_col":   pd.array(["a", "b", None, "d"], dtype="string"),  # 3.0 default for strings
    "category_col": pd.Categorical(["low", "high", "low", "medium"],
                                    categories=["low", "medium", "high"], ordered=True),
    "datetime_col": pd.to_datetime(["2025-01-01", "2025-06-15", None, "2026-01-01"]),
    "timedelta":    pd.to_timedelta(["1 day", "2 hours", None, "30 min"]),
})

print(dtype_demo)
print("\nDtypes:")
print(dtype_demo.dtypes)


### Why nullable dtypes matter

The lowercase types (`int64`, `bool`) **cannot represent missing values**. If you have integers with nulls, pandas historically silently upcasts to `float64` (because `NaN` is a float), losing the integer semantics.

The capitalized **nullable** types (`Int64`, `Boolean`, `string`, etc.) use `pd.NA` as the missing marker and preserve their type. **Use them whenever a column might have nulls.**

Interview line: "I default to nullable dtypes for any column that could have missing values, especially integer IDs, because `Int64` preserves the integer semantics while `int64` would silently upcast to float."


In [ ]:
# The integer-upcast trap.
old_way = pd.Series([1, 2, None, 4])             # pandas infers float64
new_way = pd.Series([1, 2, None, 4], dtype="Int64")  # nullable integer

print(f"Old way (no dtype hint):  dtype={old_way.dtype}, values={list(old_way)}")
print(f"New way (Int64 nullable): dtype={new_way.dtype}, values={list(new_way)}")
# Note: in the nullable version, the missing value is <NA>, not NaN.


## 2.3 `category` dtype — and when to use it

Use `category` when a string column has a **small, repeating** set of values (low cardinality). Example: city names in a 10M-row table.

Tradeoffs:
- **Memory:** category stores the unique values once and an integer code per row. A 10M-row column of city names goes from ~600MB (StringDtype) to ~10MB (category) if there are only ~50 unique cities.
- **Speed:** `groupby` on category is faster.
- **Caveat:** if cardinality is high (millions of unique values), category is *slower and bigger*. Rule of thumb: `nunique / len < 0.5` is a good candidate.

If the categories have a natural order (`low < medium < high`), pass `ordered=True` — now `<` and `>` comparisons work on the column.

In [ ]:
# Concrete memory savings demo.
n = 100_000
cities = ["Hyderabad", "Bangalore", "Mumbai", "Delhi", "Chennai", "Kolkata"]
sample = RNG.choice(cities, size=n)

df_string = pd.DataFrame({"city": pd.array(sample, dtype="string")})
df_cat    = pd.DataFrame({"city": pd.Categorical(sample)})

mem_s = df_string.memory_usage(deep=True)["city"]
mem_c = df_cat.memory_usage(deep=True)["city"]
print(f"StringDtype memory: {mem_s/1024:>8.1f} KB")
print(f"Category memory:    {mem_c/1024:>8.1f} KB")
print(f"Savings:            {mem_s/mem_c:>8.1f}x")


## 2.4 Copy-on-Write — what changed in 3.0

**The old world (pandas < 2.0):** assigning to a slice could silently modify (or not modify) the original frame depending on whether the slice was a view or a copy. The `SettingWithCopyWarning` tried to warn you but was unreliable.

**The new world (pandas 3.0):** Copy-on-Write is on by default. The rule:
- Reading from a view is fine.
- Writing through a view triggers a copy automatically. You never accidentally mutate the parent.

The practical effect: **chained assignment now does nothing** (the operation completes but doesn't affect the original). You must use `.loc[]` for assignments.

In [ ]:
# The chained-indexing trap — old code that "worked" by accident.
df = pd.DataFrame({"x": [1, 2, 3, 4], "flag": ["a", "b", "c", "d"]})

# This is the WRONG way (still wrong in 3.0 — just now silent failure instead of warning).
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    df[df["x"] > 2]["flag"] = "ZZZ"     # chained: df[mask] returns a copy in CoW
print("After chained assignment (no effect under CoW):")
print(df)

# The RIGHT way — single .loc[] call.
df.loc[df["x"] > 2, "flag"] = "ZZZ"
print("\nAfter df.loc[mask, col] = value (correct):")
print(df)


## 2.5 `loc` vs `iloc` vs `at` vs `iat`

| Accessor | Indexer type | Returns | When to use |
|---|---|---|---|
| `loc[row_label, col_label]` | labels | scalar / Series / DataFrame | most common, label-based |
| `iloc[row_pos, col_pos]` | integer positions | scalar / Series / DataFrame | when you need positional |
| `at[row_label, col_label]` | labels (single cell) | scalar | fastest scalar access |
| `iat[row_pos, col_pos]` | positions (single cell) | scalar | fastest positional scalar |

In [ ]:
df = pd.DataFrame(
    {"age": [25, 32, 47], "city": ["Hyd", "Blr", "Mum"]},
    index=["emp_1", "emp_2", "emp_3"],
)
print(df, "\n")

# Label-based slice (BOTH endpoints inclusive — different from Python slicing!).
print("df.loc['emp_1':'emp_2']  (inclusive both ends):")
print(df.loc["emp_1":"emp_2"], "\n")

# Position-based slice (end exclusive, like normal Python).
print("df.iloc[0:2]  (end exclusive):")
print(df.iloc[0:2], "\n")

# Multi-axis: row + column.
print(f"df.loc['emp_2', 'city'] = {df.loc['emp_2', 'city']}")
print(f"df.iat[1, 1]            = {df.iat[1, 1]}    <- fastest single-cell access")


**The single most-asked trap:** `df.loc[a:b]` includes both endpoints, but `df.iloc[a:b]` excludes the end. Confusing them off-by-one in an interview costs trust.

## 2.6 `read_csv` — the kwargs that matter

You'll rarely write `read_csv` from scratch in an interview, but the kwargs you mention show experience:

In [ ]:
# Conceptual cell — we don't have a CSV, but write out the call you'd use.
example_call = '''
df = pd.read_csv(
    "data.csv",
    dtype={"user_id": "Int64", "category": "category"},  # explicit dtypes save memory + bugs
    parse_dates=["created_at", "updated_at"],            # cast date columns at read time
    na_values=["", "NA", "N/A", "null", "?"],            # treat these strings as missing
    usecols=["user_id", "amount", "category", "created_at"],  # only load what you need
    chunksize=None,                                       # set to 100_000 for streaming reads
    dtype_backend="pyarrow",                              # 3.0: use Arrow backend (faster, less memory)
    on_bad_lines="warn",                                  # don't silently drop malformed rows
)
'''
print(example_call)


**Interview line per kwarg:**
- `dtype=` — "Always specify dtype for known columns. Saves memory and catches silent type-inference bugs."
- `parse_dates=` — "Cheaper to parse at read time than to convert later, and surfaces format errors immediately."
- `na_values=` — "Real-world CSVs use 'NA', 'N/A', '?', '-', and empty strings interchangeably. I always pass an explicit list."
- `usecols=` — "If the CSV is 50 columns and I need 5, loading all of them is a 10x waste."
- `chunksize=` — "For files larger than RAM, I iterate in chunks and aggregate."
- `dtype_backend="pyarrow"` — "In 3.0, the Arrow backend uses less memory and has faster string ops. The default backend is still NumPy for compatibility."


---
**Part 2 checklist:**
- [ ] Difference between `int64` and `Int64`? (nullable preserves integer semantics)
- [ ] When to use `category` dtype? (low cardinality, repeating strings)
- [ ] What CoW changed in 3.0? (chained assignment is now silent no-op; use `.loc`)
- [ ] `loc[a:b]` vs `iloc[a:b]` slicing semantics? (loc inclusive, iloc exclusive)
- [ ] 4 `read_csv` kwargs that show experience? (dtype, parse_dates, na_values, usecols)


---
# Part 3 — The First 5 Minutes Inspection Framework

The single highest-leverage section. Everything before this was tools; this is **the routine you rehearse**.

When an interviewer hands you a dataset, you have one shot to look prepared. The script below is what you run, in order, every time. If you do nothing else from this notebook, memorize this section.

## 3.1 The synthetic dataset we'll use throughout Parts 3 to 8

We'll generate an employee-churn-style dataset that has, on purpose:
- Multiple feature types (continuous, ordinal, nominal, boolean, datetime, text, ID)
- Realistic missingness patterns (MCAR, MAR, MNAR)
- Outliers
- One **leakage** column (a feature that's defined using the target)
- Some typos in a categorical
- Class imbalance in the target

This is what real interview datasets look like.

In [ ]:
def generate_employees(n=2000, seed=42):
    '''Generate a synthetic employee dataset with realistic mess.'''
    rng = np.random.default_rng(seed)

    # --- Clean signal columns ---
    employee_id = np.array([f"E{i:05d}" for i in range(n)])
    age         = rng.integers(low=22, high=60, size=n)
    tenure_yrs  = np.clip(rng.normal(loc=4.5, scale=3.0, size=n), 0, None).round(2)
    salary_inr  = (rng.lognormal(mean=11.5, sigma=0.45, size=n)).round(-2).astype(int)  # right-skewed
    satisfaction= rng.choice([1, 2, 3, 4, 5], size=n, p=[0.05, 0.10, 0.25, 0.40, 0.20])  # ordinal 1-5
    department  = rng.choice(
        ["Engineering", "Sales", "HR", "Marketing", "Finance", "Operations"],
        size=n, p=[0.40, 0.18, 0.07, 0.12, 0.10, 0.13],
    )
    job_level   = rng.choice(["L1", "L2", "L3", "L4", "L5"], size=n,
                              p=[0.20, 0.30, 0.25, 0.15, 0.10])
    remote      = rng.choice([True, False], size=n, p=[0.35, 0.65])
    join_date   = pd.to_datetime("2020-01-01") + pd.to_timedelta(
        rng.integers(low=0, high=2100, size=n), unit="D"
    )

    # --- The target: did the employee leave? ---
    # Designed so that low satisfaction + low tenure + younger age increase leave probability.
    base_logit = (
        -2.2
        + (-0.55) * (satisfaction - 3)
        + (-0.20) * (tenure_yrs - 4.5)
        + (-0.04) * (age - 35)
        + 0.7  * (np.array(department) == "Sales")
        + 0.5  * remote.astype(int)
    )
    leave_prob = 1.0 / (1.0 + np.exp(-base_logit))
    left = rng.binomial(1, leave_prob, size=n).astype(bool)

    # --- The LEAKAGE column: "exit_reason" is only filled for leavers ---
    exit_reasons = ["pay", "manager", "growth", "personal", "wlb", None]
    exit_reason  = np.where(
        left,
        rng.choice(exit_reasons[:-1], size=n),
        None,
    )

    # --- Add typos to department (real-world data quality issue) ---
    typo_idx = rng.choice(n, size=int(0.02 * n), replace=False)
    typo_map = {"Engineering": "engineering", "Sales": "sales ", "HR": " HR"}
    department = department.copy()
    for i in typo_idx:
        if department[i] in typo_map:
            department[i] = typo_map[department[i]]

    # --- Free-text exit comment (only for leavers, and only ~60% of leavers wrote one) ---
    comment_templates = [
        "Manager was {tone}. {action}.",
        "Pay below market. {detail}.",
        "Looking for {goal}.",
        "Personal reasons.",
        "Better opportunity at another company {detail}.",
        "WLB issues, {detail}.",
        "",
    ]
    tones    = ["unsupportive", "micromanaging", "great", "okay", "absent"]
    actions  = ["No growth path", "Too much politics", "Decided to move on", "Pursuing MBA"]
    details  = ["got 40% hike elsewhere", "burnout was real", "joining a startup", "moving cities"]
    goals    = ["growth", "remote work", "higher pay", "better culture"]

    exit_comment = np.array(["" for _ in range(n)], dtype=object)
    leaver_idx = np.where(left)[0]
    write_comment_idx = rng.choice(leaver_idx, size=int(0.6 * len(leaver_idx)), replace=False)
    for i in write_comment_idx:
        tmpl = rng.choice(comment_templates)
        exit_comment[i] = tmpl.format(
            tone=rng.choice(tones), action=rng.choice(actions),
            detail=rng.choice(details), goal=rng.choice(goals),
        )

    # --- Build frame ---
    df = pd.DataFrame({
        "employee_id":   employee_id,
        "age":           age,
        "tenure_yrs":    tenure_yrs,
        "salary_inr":    salary_inr,
        "satisfaction":  satisfaction,
        "department":    pd.array(department, dtype="string"),
        "job_level":     pd.array(job_level, dtype="string"),
        "remote":        remote,
        "join_date":     join_date,
        "exit_reason":   pd.array(exit_reason, dtype="string"),
        "exit_comment":  pd.array(exit_comment, dtype="string"),
        "left":          left,
    })

    # --- Add realistic missingness ---
    # MCAR (Missing Completely At Random): drop 3% of salary at random.
    mask_mcar = rng.random(n) < 0.03
    df.loc[mask_mcar, "salary_inr"] = pd.NA

    # MAR (Missing At Random): satisfaction is more often missing for new joiners.
    new_joiner = df["tenure_yrs"] < 1.0
    mask_mar = (rng.random(n) < 0.20) & new_joiner
    df.loc[mask_mar, "satisfaction"] = pd.NA

    # MNAR (Missing Not At Random): high earners hide their salary more often.
    mask_mnar = (rng.random(n) < 0.15) & (df["salary_inr"].fillna(0) > 200_000)
    df.loc[mask_mnar, "salary_inr"] = pd.NA

    # --- Add 8 exact duplicates (data plumbing bug simulation) ---
    dup_idx = rng.choice(n, size=8, replace=False)
    df = pd.concat([df, df.iloc[dup_idx]], ignore_index=True)

    # Force salary to nullable Int64 so it survives the NAs.
    df["salary_inr"]   = df["salary_inr"].astype("Int64")
    df["satisfaction"] = df["satisfaction"].astype("Int64")

    return df


# Generate the dataset we'll use through the rest of the notebook.
emp = generate_employees(n=2000, seed=42)
print(f"Generated dataset shape: {emp.shape}")


## 3.2 Step 1 — `df.shape` (scale check)

In [ ]:
print(f"Shape: {emp.shape}")
print(f"Rows:    {emp.shape[0]:>6}")
print(f"Columns: {emp.shape[1]:>6}")


**What to say:** "We have 2008 rows and 12 columns. Small enough to fit in memory comfortably, large enough to compute meaningful statistics. If the dataset were >1M rows I'd consider sampling for the EDA pass, then run the final pipeline on the full set."


## 3.3 Step 2 — `head()` and `sample()` (what does a row mean?)

In [ ]:
# Always start with head, then sample. head shows row 0 (often special), sample shows the middle.
emp.head()


In [ ]:
# Sample is critical: head() can be misleading (sorted, headers, special first rows).
# Always also look at a random sample.
emp.sample(n=5, random_state=42)


**What to ask the interviewer right now:** "What does one row represent? Is it one employee at a point in time, or one employee per month? Are inactive employees included?" — this is the **unit of analysis** question. If you skip it, every downstream analysis is on shaky ground.

## 3.4 Step 3 — `df.info()` (types, nulls, memory)

In [ ]:
emp.info(memory_usage="deep")


**Read this output line by line:**
- **Non-Null Count**: which columns have missing values? `salary_inr` and `satisfaction` are missing some; `exit_reason` and `exit_comment` are very sparse (only filled for leavers).
- **Dtype**: are all dtypes sensible? `employee_id` should probably be `string` (it's an object column). Nullable `Int64` for the salary and satisfaction is correct.
- **Memory**: 2008 rows × 12 cols using ~XXX KB. For a million-row scale-up: this would be ~XXX MB.

**Trap question:** "Why is the `object` dtype on `employee_id` a problem?" Because object dtype stores Python pointers, not packed strings. For a million-row table it'll be ~5x larger than `StringDtype`.

## 3.5 Step 4 — `describe()` (quick statistics)

In [ ]:
# Default: numeric columns only.
emp.describe()


In [ ]:
# include='all' adds object/category/string columns too — shows top value and frequency.
emp.describe(include="all")


**What to look for in `describe`:**
- **count vs total rows** — if count < total, that column has missing values.
- **mean vs median** — if mean >> median (or vice versa), the distribution is skewed. `salary_inr` likely shows this.
- **std** — gives a sense of spread. Compare to mean (coefficient of variation).
- **min and max** — sanity check for impossible values (negative age, salary of 0, dates in the future).
- **For string columns:** `unique` (cardinality), `top` (mode), `freq` (mode frequency).

## 3.6 Step 5 — `nunique()` and `isna().sum()` (uniqueness + missingness)

In [ ]:
# Uniqueness per column — tells you which columns are IDs, categoricals, or continuous.
uniqueness = pd.DataFrame({
    "nunique":  emp.nunique(),
    "n_rows":   len(emp),
    "unique_pct": (emp.nunique() / len(emp) * 100).round(2),
    "dtype":    emp.dtypes.astype(str),
})
uniqueness


In [ ]:
# Missingness per column — both count and percentage.
missingness = pd.DataFrame({
    "n_missing":   emp.isna().sum(),
    "pct_missing": (emp.isna().mean() * 100).round(2),
}).sort_values("pct_missing", ascending=False)
missingness


**Reading uniqueness:**
- `unique_pct ≈ 100%` → ID column (or free text, or near-unique continuous)
- `unique_pct < 5%` → categorical or ordinal candidate
- In between → continuous numeric or high-cardinality categorical

**Reading missingness:**
- `exit_reason` and `exit_comment` are ~85% missing — but **not really missing**, just only-filled-for-leavers. This is "structural missingness," not data quality.
- `salary_inr` ~5% missing — real data quality issue, needs investigation.
- `satisfaction` ~2% missing — small, but check if missingness correlates with anything (MAR).

## 3.7 Step 6 — Classify every column into a feature type

In [ ]:
# Build the feature-type table manually. This is what you'd write on a whiteboard.
feature_table = pd.DataFrame([
    ("employee_id",  "ID",        "string",   "unique key, drop before modeling"),
    ("age",          "continuous","int64",    "22-60, roughly uniform"),
    ("tenure_yrs",   "continuous","float64",  "right-skewed, clip-at-zero"),
    ("salary_inr",   "continuous","Int64",    "right-skewed (log-normal), has 5% missing"),
    ("satisfaction", "ordinal",   "Int64",    "1-5 scale, has ordering"),
    ("department",   "nominal",   "string",   "6 categories + typos to clean"),
    ("job_level",    "ordinal",   "string",   "L1-L5 has ordering; encode as ordered category"),
    ("remote",       "boolean",   "bool",     "balanced ~35/65"),
    ("join_date",    "datetime",  "datetime", "2020-01-01 to ~2025, derive tenure if not given"),
    ("exit_reason",  "nominal",   "string",   "only filled for leavers - LEAKAGE risk if used as feature"),
    ("exit_comment", "free_text", "string",   "only filled for leavers - LEAKAGE risk"),
    ("left",         "target",    "bool",     "binary target, imbalanced ~75/25"),
], columns=["column", "feature_type", "dtype", "notes"])

feature_table


## 3.8 Step 7 — The 10 questions to ask the interviewer

This is the differentiating section. **The candidate who asks these questions before touching code looks senior.** Memorize them.

1. **What is one row?** (the unit of analysis)
2. **What is the prediction target, and how is it defined operationally?** ("Did they leave" — within what window? Voluntary or involuntary? Including transfers?)
3. **Is this a snapshot or a time series panel?** (one row per employee, or one row per employee-month?)
4. **How was the data sampled?** (all employees? a stratified sample? only those who joined after some date?)
5. **What's the time range, and are there structural breaks?** (e.g., COVID, a re-org, an acquisition — these change the data-generating process)
6. **Are any columns derived from the target?** (the **leakage** question — `exit_reason` here)
7. **Are duplicates expected, accidental, or forbidden?** (one row per employee → exact dupes are bugs)
8. **What's the labeling process and known error rate?** (was "left" auto-detected from HR systems, or hand-labeled?)
9. **Which columns are user-editable vs system-generated?** (user-editable fields are noisier and have selection bias)
10. **What does missing mean for each column?** ("not applicable," "not yet recorded," "user refused," or a data plumbing bug — each requires a different fix)

**Pro move:** for any dataset where the answer to (5) involves COVID or any major shift, immediately split the data on the break date and run a quick distribution-shift check.

## 3.9 The whole framework as a single function

In [ ]:
def quick_eda(df: pd.DataFrame) -> None:
    '''Run the 7-step inspection on a dataframe. The function you'd paste into any new notebook.'''
    print("=" * 72)
    print(f"STEP 1 — Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print("=" * 72)

    print("\nSTEP 2 — Head (first 3 rows):")
    print(df.head(3))
    print("\n         Sample (random 3 rows):")
    print(df.sample(min(3, len(df)), random_state=0))

    print("\n" + "=" * 72)
    print("STEP 3 — Types and non-null counts")
    print("=" * 72)
    df.info(memory_usage="deep")

    print("\n" + "=" * 72)
    print("STEP 4 — describe(include='all')")
    print("=" * 72)
    print(df.describe(include="all").T)  # transpose for readability when many columns

    print("\n" + "=" * 72)
    print("STEP 5 — Uniqueness + Missingness")
    print("=" * 72)
    summary = pd.DataFrame({
        "dtype":       df.dtypes.astype(str),
        "n_unique":    df.nunique(),
        "unique_pct":  (df.nunique() / len(df) * 100).round(1),
        "n_missing":   df.isna().sum(),
        "miss_pct":    (df.isna().mean() * 100).round(1),
    })
    print(summary)

    print("\n" + "=" * 72)
    print("STEP 6 & 7 — Feature classification + questions to ask")
    print("=" * 72)
    print("(Do this part by reading the output above and writing on a whiteboard.)")
    print("Always ask: unit of analysis? target definition? snapshot vs panel?")
    print("            sampling? time range? leakage? duplicate semantics?")
    print("            label process? user vs system fields? missing semantics?")


# Run it on the employee dataset.
quick_eda(emp)


---
**Part 3 checklist:**
- [ ] Can you recite the 7 inspection steps in order?
- [ ] Can you recite at least 7 of the 10 interviewer-questions?
- [ ] When you see `unique_pct ≈ 100%`, what do you conclude? (ID column or free text)
- [ ] What's "structural missingness" vs "data quality missingness"? (`exit_reason` vs `salary_inr` in this dataset)
- [ ] What's the leakage column in this dataset and why? (`exit_reason` — defined by the target)


---
# Part 4 — Feature-type taxonomy and per-type checks

For each feature type, there's a **standard set of checks** an interviewer expects you to run. Below is the cheat sheet. Memorize this table — it's what you should mentally walk through for each column.

## 4.1 The taxonomy table

| Feature type | Examples | What to check | What to plot | Question to ask |
|---|---|---|---|---|
| **ID / key** | user_id, transaction_id | uniqueness, duplicates, format consistency | none | "should this be unique? what defines the join key?" |
| **Continuous numeric** | salary, age, temperature | range, distribution shape, skew, outliers, units | histogram, boxplot | "what's the valid range? log-transform candidate?" |
| **Nominal categorical** | city, department, color | cardinality, mode, rare categories, typos | bar chart of counts | "how many categories? are rare ones meaningful or noise?" |
| **Ordinal categorical** | satisfaction (1-5), level (L1-L5) | encoding decision, distribution per rank | ordered bar | "is the spacing between ranks equal or just ordered?" |
| **Boolean** | is_active, remote | class balance | bar/pie | "what does True mean operationally?" |
| **Datetime** | join_date, event_time | range, gaps, seasonality, timezone | timeline plot | "what timezone? local or UTC? gaps expected?" |
| **Geo** | lat/lon, country | valid ranges, missingness, granularity | map | "what's the spatial resolution? is it user-reported?" |
| **Free text** | reviews, comments | length distribution, vocab, quality flags | length histogram, wordcloud | "what's the source? is it user-generated?" |

## 4.2 The decision tree for "what type is this column?"

Use this in order:

1. **Is it a unique identifier?** (nunique ≈ n_rows, doesn't relate to other features) → **ID**
2. **Is it a datetime or parses as one?** → **Datetime**
3. **Is it stored as numeric but has only 2 unique values?** → **Boolean**
4. **Is it numeric with a small set of discrete values (say, <15) AND has natural ordering?** → **Ordinal**
5. **Is it numeric with a small set of discrete values but no natural ordering?** → **Nominal-encoded-as-numeric** (rare but happens — like ICD-10 codes)
6. **Is it numeric with many values, continuous-looking?** → **Continuous**
7. **Is it a string with low cardinality (say, nunique/n < 5%)?** → **Nominal**
8. **Is it a string with high average length (say, >50 chars) and high cardinality?** → **Free text**
9. **Otherwise (string, medium cardinality, short)** → ambiguous, look at it

In [ ]:
def auto_classify(df: pd.DataFrame) -> pd.DataFrame:
    '''Auto-classify columns by the decision tree above. Output is a starting point, not gospel.'''
    rows = []
    n = len(df)

    for col in df.columns:
        s = df[col]
        n_unique = s.nunique(dropna=True)
        unique_pct = n_unique / n if n > 0 else 0
        dtype = str(s.dtype)

        # Use pandas type-check helpers — they handle both NumPy and pandas extension dtypes.
        is_dt   = pd.api.types.is_datetime64_any_dtype(s)
        is_bool = pd.api.types.is_bool_dtype(s)
        is_num  = pd.api.types.is_numeric_dtype(s) and not is_bool
        is_str  = pd.api.types.is_string_dtype(s) and not is_num and not is_dt

        # Default
        guess = "unknown"

        # 1. ID
        if unique_pct > 0.95:
            guess = "ID (or free text)"
            # disambiguate ID vs free text using string length if applicable
            if is_str:
                avg_len = s.dropna().astype(str).str.len().mean()
                if avg_len and avg_len > 50:
                    guess = "free_text"

        # 2. Datetime
        elif is_dt:
            guess = "datetime"

        # 3. Boolean
        elif is_bool or (is_num and n_unique == 2):
            guess = "boolean"

        # 4 & 5. Small discrete numeric
        elif is_num and n_unique <= 15:
            guess = "ordinal_or_nominal_numeric"

        # 6. Continuous
        elif is_num:
            guess = "continuous"

        # 7. Nominal (low-cardinality string)
        elif is_str and unique_pct < 0.05:
            guess = "nominal"

        # 8. Free text (long strings)
        elif is_str:
            avg_len = s.dropna().astype(str).str.len().mean()
            guess = "free_text" if (avg_len and avg_len > 50) else "nominal_or_text"

        rows.append({
            "column":    col,
            "dtype":     dtype,
            "n_unique":  n_unique,
            "unique_pct": round(unique_pct * 100, 2),
            "auto_type":  guess,
        })

    return pd.DataFrame(rows)


# Apply to the employee dataset.
auto_classify(emp)


**Caveat:** auto-classification is a *first pass*. You always confirm with `value_counts()` and a quick sanity check — code can't know that a numeric column with 5 unique values is ordinal (`satisfaction`) vs nominal (`error_code`). That's a domain question.

---
# Part 5 — Univariate analysis

For each column, look at it **on its own** first. Multivariate analysis is later. This section: what to run for each feature type, and how to read the output.

## 5.1 Continuous: `describe`, distribution shape, outliers

In [ ]:
# Focus on a continuous column: salary.
salary = emp["salary_inr"].dropna()

print("describe():")
print(salary.describe())

# Skewness and kurtosis quantify "non-normality."
# Skewness > 0  -> right tail heavier (positive skew, common for salary, income, prices)
# Skewness < 0  -> left tail heavier (negative skew, e.g. age-at-death)
# Kurtosis > 0  -> heavier tails than normal (more outliers)
# Both ~ 0     -> roughly normal
print(f"\nSkewness:  {stats.skew(salary):.3f}    (>0 = right-tailed)")
print(f"Kurtosis:  {stats.kurtosis(salary):.3f}    (>0 = heavy tails)")

# Normality test — Shapiro-Wilk is the standard but only works up to n=5000.
# For larger samples, use D'Agostino-Pearson (normaltest).
stat, p_value = stats.normaltest(salary)
print(f"\nD'Agostino-Pearson normality test: stat={stat:.2f}, p={p_value:.2e}")
print("p < 0.05 means we reject the null (data is NOT normal).")


In [ ]:
# Visualize distribution: histogram + KDE side by side with a boxplot.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: histogram with KDE overlay.
sns.histplot(salary, kde=True, ax=axes[0], bins=40, color="steelblue")
axes[0].axvline(salary.mean(),   color="red",    linestyle="--", label=f"mean   ₹{salary.mean():,.0f}")
axes[0].axvline(salary.median(), color="orange", linestyle="--", label=f"median ₹{salary.median():,.0f}")
axes[0].set_title("Salary distribution (right-skewed)")
axes[0].set_xlabel("Salary (INR)")
axes[0].legend()

# Right: boxplot — outliers are dots outside the whiskers.
sns.boxplot(x=salary, ax=axes[1], color="steelblue")
axes[1].set_title("Salary boxplot (outliers on the right)")
axes[1].set_xlabel("Salary (INR)")

plt.tight_layout()
plt.show()


**Read it:** the histogram shows a clear right skew (long tail to the right). Mean (red) is to the right of the median (orange) — the classic signature of right skew. The boxplot's whiskers extend further on the right, and outlier dots beyond the upper whisker confirm heavy upper tail.

**What to say:** "Salary is right-skewed, mean > median, so I'd consider a log transform if I want a roughly symmetric input for any linear model. The upper tail outliers are probably real (executives), not data errors — but I'd confirm with the interviewer."


## 5.2 Outlier detection — three methods

In [ ]:
def detect_outliers_iqr(series, k=1.5):
    '''
    Tukey's IQR method: outliers are below Q1 - k*IQR or above Q3 + k*IQR.
    k=1.5 (standard), k=3.0 (extreme outliers only).
    Robust to non-normal data — recommended default.
    '''
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - k * iqr, q3 + k * iqr
    mask = (series < lower) | (series > upper)
    return mask, (lower, upper)


def detect_outliers_zscore(series, threshold=3.0):
    '''
    Z-score: outliers are |z| > threshold. Assumes (approximately) normal data.
    Bad for skewed distributions — the mean and std are themselves pulled by outliers.
    '''
    z = (series - series.mean()) / series.std()
    return z.abs() > threshold


def detect_outliers_modified_zscore(series, threshold=3.5):
    '''
    Modified z-score using median and MAD (Median Absolute Deviation).
    Robust to outliers in the data — recommended for skewed distributions.
    Threshold 3.5 corresponds to roughly the same alpha as z=3 for normal data.
    '''
    median = series.median()
    mad = (series - median).abs().median()
    # 0.6745 makes MAD a consistent estimator of std for normal data.
    modified_z = 0.6745 * (series - median) / mad
    return modified_z.abs() > threshold


salary_clean = emp["salary_inr"].dropna().astype(float)

mask_iqr, (lo, hi) = detect_outliers_iqr(salary_clean)
mask_z   = detect_outliers_zscore(salary_clean)
mask_mz  = detect_outliers_modified_zscore(salary_clean)

print(f"IQR method            : {mask_iqr.sum():>4} outliers   (range: {lo:,.0f} to {hi:,.0f})")
print(f"Z-score (|z|>3)       : {mask_z.sum():>4} outliers")
print(f"Modified Z (|z|>3.5)  : {mask_mz.sum():>4} outliers")


**When to use which:**
- **IQR** is the default. Robust, distribution-free, what boxplots use.
- **Z-score** assumes normality. Use only on roughly-normal data. Salary is skewed, so z-score under-counts here.
- **Modified z-score** is the robust alternative to z-score. Uses median and MAD, both robust to outliers themselves.

**Critical interview point:** "An outlier in the data is not the same as an error. A CEO's ₹10cr salary is an outlier but not wrong. Always ask whether outliers should be removed, capped (winsorized), or kept as-is."


## 5.3 Skew correction — log, Box-Cox, Yeo-Johnson

In [ ]:
# When you have a right-skewed positive variable, log transform often makes it roughly normal.
salary_clean = emp["salary_inr"].dropna().astype(float)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(salary_clean, kde=True, ax=axes[0], bins=40, color="steelblue")
axes[0].set_title(f"Original salary (skew = {stats.skew(salary_clean):.2f})")

sns.histplot(np.log(salary_clean), kde=True, ax=axes[1], bins=40, color="darkgreen")
axes[1].set_title(f"log(salary) (skew = {stats.skew(np.log(salary_clean)):.2f})")

plt.tight_layout()
plt.show()


**The transform decision tree:**
- **Strictly positive + right-skewed** → `log` (or `log1p` if zeros are present)
- **Strictly positive + skew is unknown direction** → `Box-Cox` (auto-picks the right power transform)
- **Can be zero or negative** → `Yeo-Johnson` (Box-Cox generalized to handle non-positive values)
- **Heavy outliers in both tails** → `RankGauss` or `QuantileTransformer(output='normal')`

In sklearn: `from sklearn.preprocessing import PowerTransformer` with `method='box-cox'` or `'yeo-johnson'`.


## 5.4 Categorical: `value_counts`, mode dominance, rare categories

In [ ]:
# For categorical columns, value_counts is the bread and butter.
# Always pass dropna=False so missing shows up explicitly.
# normalize=True converts to proportions instead of counts.

print("Department counts:")
print(emp["department"].value_counts(dropna=False))

print("\nDepartment proportions:")
print(emp["department"].value_counts(normalize=True, dropna=False).round(3))


**Notice the typos:** `engineering`, `sales `, ` HR` — same logical categories with different surface forms. This is **the** most common data quality issue with strings. Two fixes:

In [ ]:
# Quick fix: normalize whitespace and case.
emp["department_clean"] = (
    emp["department"]
    .str.strip()        # remove leading/trailing whitespace
    .str.title()        # consistent case ("engineering" -> "Engineering", "HR" -> "Hr" though!)
)
# Title-case fails for "HR" -> "Hr". Real cleanup needs a known-good mapping.
print("After strip + title (HR becomes Hr — title case is naive):")
print(emp["department_clean"].value_counts())


In [ ]:
# Better fix: explicit mapping to canonical values.
canonical_map = {
    "Engineering": "Engineering", "engineering": "Engineering",
    "Sales": "Sales", "sales": "Sales",
    "HR": "HR", "hr": "HR",
    "Marketing": "Marketing", "Finance": "Finance", "Operations": "Operations",
}
# Apply with case-folding pre-step.
emp["department_clean"] = emp["department"].str.strip().map(
    lambda v: canonical_map.get(v, canonical_map.get(v.lower() if isinstance(v, str) else v, v))
)
print("After explicit canonical map:")
print(emp["department_clean"].value_counts())


**Rare-category strategy:** if a category has fewer than (say) 50 occurrences out of 2000, consider:
- **Bucket into "Other"** — simplest, loses signal
- **Target encoding** — replace with the mean target rate for that category (careful: needs out-of-fold to avoid leakage)
- **Hashing trick** — for high cardinality, hash to a smaller fixed bucket count

For this dataset, no category is below 50, so no action needed.

## 5.5 Ordinal: respecting the order

In [ ]:
# satisfaction is 1-5 with natural ordering. Encode as ordered category to unlock < and > semantics.
emp["satisfaction_cat"] = pd.Categorical(
    emp["satisfaction"],
    categories=[1, 2, 3, 4, 5],
    ordered=True,
)
print("Ordered satisfaction:")
print(emp["satisfaction_cat"].value_counts().sort_index())
print(f"\nIs ordered? {emp['satisfaction_cat'].cat.ordered}")
print(f"5 > 3? {(emp['satisfaction_cat'].iloc[0] if pd.notna(emp['satisfaction_cat'].iloc[0]) else 5) > 3}")


In [ ]:
# Plot ordinal distributions with ORDER preserved.
fig, ax = plt.subplots(figsize=(8, 4))
sat_counts = emp["satisfaction"].value_counts().sort_index()
ax.bar(sat_counts.index, sat_counts.values, color="steelblue")
ax.set_xlabel("Satisfaction (1=low, 5=high)")
ax.set_ylabel("Count")
ax.set_title("Satisfaction distribution (ordinal — order matters)")
plt.show()


**Interview gotcha:** if you bar-chart an ordinal column without sorting by the category order, you can end up with the chart in alphabetical order ("L1, L2, L3, L4, L5" is fine; "five, four, one, three, two" is a disaster). Always `sort_index()` for ordinal counts.

## 5.6 Long-tail detection with `value_counts().cumsum()`

In [ ]:
# Realistic long-tail: simulate a "product_views" column.
views = pd.Series(RNG.zipf(a=1.5, size=10000))   # Zipf distribution — classic long tail
vc = views.value_counts()

# Cumulative coverage: what fraction of TOTAL views do the top-K products account for?
cum_coverage = vc.cumsum() / vc.sum()
print(f"Top 10 products cover  {cum_coverage.iloc[9]*100:.1f}% of all views")
print(f"Top 50 products cover  {cum_coverage.iloc[49]*100:.1f}% of all views")
print(f"Top 100 products cover {cum_coverage.iloc[99]*100:.1f}% of all views")

# Plot — log scale on x-axis makes the long tail visible.
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(1, len(vc) + 1), cum_coverage.values, color="steelblue")
ax.set_xscale("log")
ax.axhline(0.8, color="red", linestyle="--", label="80% coverage")
ax.set_xlabel("Top-K products (log scale)")
ax.set_ylabel("Cumulative coverage of all events")
ax.set_title("Long-tail check: how many categories cover 80% of events?")
ax.legend()
plt.show()


**Interview line:** "Most real categorical columns are long-tailed (Zipf's law). I check what fraction of unique categories covers 80% of events — that's my decision point for whether to bucket the tail into 'Other'."


---
**Part 5 checklist:**
- [ ] When to use IQR vs z-score vs modified z-score for outliers?
- [ ] When to use log vs Box-Cox vs Yeo-Johnson?
- [ ] What does `value_counts(normalize=True, dropna=False)` give you?
- [ ] How do you encode an ordinal so `<` and `>` work? (`pd.Categorical(..., ordered=True)`)
- [ ] Why do you `sort_index()` before bar-charting an ordinal?


---
# Part 6 — Bivariate and multivariate analysis

Once you've inspected each column on its own, you look at **relationships**. The matrix below tells you which technique to use for each (predictor, target) combination:

| Predictor → Target ↓ | Continuous | Categorical | Binary |
|---|---|---|---|
| **Continuous** | Pearson / Spearman correlation, scatter | Boxplot per category, ANOVA | Point-biserial correlation |
| **Categorical** | Boxplot, Mann-Whitney | Cramér's V, χ² test, contingency table | Cramér's V, χ² test |
| **Binary** | Point-biserial, logistic intuition | Cramér's V | Phi coefficient |

For non-linear relationships, **Mutual Information** is the universal fallback.

## 6.1 Continuous × Continuous — Pearson, Spearman, Kendall

In [ ]:
# Look at the relationship between tenure and salary.
sub = emp[["tenure_yrs", "salary_inr", "age", "satisfaction"]].dropna()

# Three correlation methods:
# - Pearson:  linear correlation, assumes interval data, sensitive to outliers
# - Spearman: rank correlation, captures any monotonic relationship, robust to outliers
# - Kendall:  rank correlation based on concordant/discordant pairs, more conservative
# Rule of thumb: report Pearson AND Spearman. If they differ a lot, you have non-linearity or outliers.

print("Pearson (linear):")
print(sub.corr(method="pearson").round(3))

print("\nSpearman (monotonic, rank-based):")
print(sub.corr(method="spearman").round(3))


In [ ]:
# Visualize the correlation matrix as a heatmap.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(sub.corr(method="pearson"), annot=True, fmt=".2f", cmap="coolwarm",
            center=0, vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title("Pearson correlation")

sns.heatmap(sub.corr(method="spearman"), annot=True, fmt=".2f", cmap="coolwarm",
            center=0, vmin=-1, vmax=1, ax=axes[1])
axes[1].set_title("Spearman correlation")

plt.tight_layout()
plt.show()


**The single most-important interview line:** "Correlation only captures linear relationships (Pearson) or monotonic ones (Spearman). Zero correlation does **not** mean independence — a parabola y = x² has zero Pearson correlation with x but they're perfectly dependent. For non-linear relationships I'd use mutual information or a nonparametric method like Hoeffding's D."

**Bonus gotcha:** correlation of 1.0 (or -1.0) between two features is a **leakage smell** — one feature is literally a function of the other. Always investigate.

## 6.2 Scatter plot with marginal histograms

In [ ]:
# Tenure vs salary, colored by whether the employee left.
g = sns.jointplot(
    data=emp.dropna(subset=["salary_inr"]),
    x="tenure_yrs", y="salary_inr",
    hue="left",
    kind="scatter",
    height=6, alpha=0.5,
)
g.figure.suptitle("Tenure vs Salary, colored by leave status", y=1.02)
plt.show()


## 6.3 Continuous × Categorical — boxplots + ANOVA / Kruskal-Wallis

In [ ]:
# Does salary differ by department? Use a boxplot — fastest way to see it.
fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(
    data=emp.dropna(subset=["salary_inr"]),
    x="department_clean", y="salary_inr",
    ax=ax,
)
ax.set_title("Salary distribution by department")
ax.set_xlabel("Department")
ax.set_ylabel("Salary (INR)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


In [ ]:
# Statistical test: is mean salary different across departments?
# One-way ANOVA (parametric, assumes normality + equal variances).
# Kruskal-Wallis (non-parametric, only assumes ordinal scale) — safer for skewed data like salary.

groups = [
    g["salary_inr"].dropna().values
    for _, g in emp.groupby("department_clean", observed=True)
]

f_stat, anova_p = stats.f_oneway(*groups)
h_stat, kw_p    = stats.kruskal(*groups)

print(f"ANOVA          : F={f_stat:.2f}, p={anova_p:.2e}")
print(f"Kruskal-Wallis : H={h_stat:.2f}, p={kw_p:.2e}")
print("\np < 0.05 -> at least one group's distribution differs from the rest.")
print("For salary (skewed), trust Kruskal-Wallis over ANOVA.")


## 6.4 Categorical × Categorical — contingency table + Cramér's V

In [ ]:
# Contingency table: department vs leave status.
contingency = pd.crosstab(emp["department_clean"], emp["left"])
print("Contingency table (counts):")
print(contingency)

# Normalize by row to see leave RATE per department.
print("\nRow-normalized (leave rate per department):")
print(pd.crosstab(emp["department_clean"], emp["left"], normalize="index").round(3))


In [ ]:
def cramers_v(x, y):
    '''
    Cramér's V: effect-size measure for two categorical variables.
    Range [0, 1]: 0 = independent, 1 = perfect association.
    Interpretation guide (Cohen):
      < 0.1  : negligible
      0.1-0.3: weak
      0.3-0.5: moderate
      > 0.5  : strong
    Pearson correlation is for numeric; Cramér's V is its categorical cousin.
    '''
    confusion = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(confusion, correction=False)[0]
    n = confusion.values.sum()
    r, k = confusion.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1)))


print(f"Cramér's V (department × left): {cramers_v(emp['department_clean'], emp['left']):.3f}")
print(f"Cramér's V (job_level × left):  {cramers_v(emp['job_level'], emp['left']):.3f}")
print(f"Cramér's V (remote × left):     {cramers_v(emp['remote'], emp['left']):.3f}")


**Why Cramér's V matters:** most candidates know Pearson correlation but go silent on categorical × categorical associations. Saying "I used Cramér's V to quantify the association between department and churn" is instant signal that you've done EDA before.

## 6.5 Binary × Continuous — point-biserial correlation

In [ ]:
# Point-biserial: Pearson correlation when one variable is binary (0/1) and the other continuous.
# It's literally Pearson under the hood; the name just signals the binary-continuous pairing.

mask = emp["salary_inr"].notna()
r_pb, p = stats.pointbiserialr(
    emp.loc[mask, "left"].astype(int),
    emp.loc[mask, "salary_inr"].astype(float),
)
print(f"Point-biserial corr (left × salary):  r = {r_pb:.3f}, p = {p:.2e}")
print("Negative -> higher salary associated with lower churn.")


## 6.6 Mutual Information — the non-linear fallback

In [ ]:
# Mutual information captures ANY relationship, not just linear or monotonic.
# scikit-learn has it for both regression and classification targets.
from sklearn.feature_selection import mutual_info_classif

# Quick-and-dirty: encode categoricals to ints (sklearn MI wants numeric).
X = pd.DataFrame({
    "age":         emp["age"],
    "tenure_yrs":  emp["tenure_yrs"],
    "salary_inr":  emp["salary_inr"].fillna(emp["salary_inr"].median()).astype(float),
    "satisfaction":emp["satisfaction"].fillna(emp["satisfaction"].median()).astype(int),
    "remote":      emp["remote"].astype(int),
    "department":  pd.Categorical(emp["department_clean"]).codes,
    "job_level":   pd.Categorical(emp["job_level"]).codes,
})
y = emp["left"].astype(int)

mi = mutual_info_classif(X, y, discrete_features=[3, 4, 5, 6], random_state=42)
mi_table = pd.DataFrame({"feature": X.columns, "MI": mi}).sort_values("MI", ascending=False)
print("Mutual information with 'left':")
print(mi_table.to_string(index=False))


**When to reach for MI:**
- Suspicion of non-linear or non-monotonic relationships
- Mixed feature types (some continuous, some categorical) and you want one consistent ranking
- Feature selection before tree-based models (since MI mirrors what trees do)

**Catch:** MI estimates from small samples are noisy and biased upward. Don't over-interpret tiny differences.

## 6.7 Target leakage hunt

This is the most expensive bug in ML. A feature that's defined *using* the target, or known only *because* the target happened, will give amazing training metrics and crash on real data.

**The hunt:** for every feature, ask 'could this only have been recorded after the prediction time?' Or: 'if I gave this feature to a model, would it solve the problem so well that the model is suspiciously perfect?'

In [ ]:
# Run the leakage check on our employee dataset.
# `exit_reason` is null for non-leavers, populated for leavers — by construction.

print("Is exit_reason filled when left=True?")
print(emp.groupby("left")["exit_reason"].apply(lambda s: s.notna().mean()).round(3))

# Check the "predictive power" of just knowing whether exit_reason is null:
leakage_check = (emp["exit_reason"].notna() == emp["left"]).mean()
print(f"\n'exit_reason is not null' equals 'left' in {leakage_check*100:.1f}% of rows.")
print("That's the leakage smell — a feature that perfectly tracks the target.")


**The general leakage tests to run:**
1. For each feature, plot `feature_is_null` vs target. A massive correlation = leakage flag.
2. Train a quick tree model on each feature alone. AUC > 0.95 = leakage flag.
3. Check timestamps: any feature recorded *after* the target event time is leakage.


---
**Part 6 checklist:**
- [ ] Pearson vs Spearman vs Kendall — when to use each?
- [ ] What does "correlation = 0" not imply? (independence — only linear/monotonic)
- [ ] Cramér's V — what's it for and what's the range?
- [ ] Point-biserial — what setup? (binary × continuous, but it's just Pearson under the hood)
- [ ] Three ways to detect target leakage?


---
# Part 7 — Missing data deep dive

Missing data is **never just missing**. There are three patterns, and which one you have determines which imputation strategy is unbiased.

## 7.1 MCAR vs MAR vs MNAR

| Pattern | Definition | Example | Safe to drop? | Safe to impute? |
|---|---|---|---|---|
| **MCAR** (Missing Completely At Random) | Missingness is independent of everything. | Lab tech randomly drops a tube. | Yes (just loses power) | Yes (any method) |
| **MAR** (Missing At Random) | Missingness depends on **observed** variables, not on the missing value itself. | Income missing more often for younger respondents (and we observe age). | Risky (loses signal) | Yes if you condition on the observed cause |
| **MNAR** (Missing Not At Random) | Missingness depends on the **missing value itself**. | High earners refuse to disclose salary. | No (biased) | Hard — needs a model of the missingness mechanism |

You **cannot tell from data alone** whether missingness is MAR or MNAR — both look the same in the available data. You have to **reason from the domain**.

In [ ]:
# Recall our employee dataset was built with three patterns:
# 1. salary_inr 3% MCAR (random drops)
# 2. satisfaction 20% × tenure<1yr (MAR — depends on observed tenure)
# 3. salary_inr 15% × salary>200k (MNAR — depends on the missing value itself)

# Diagnostic 1: missingness counts and rates.
miss_summary = emp.isna().sum().to_frame("n_missing")
miss_summary["pct_missing"] = (emp.isna().mean() * 100).round(2)
miss_summary["dtype"] = emp.dtypes.astype(str)
print(miss_summary.sort_values("pct_missing", ascending=False))


## 7.2 Missingness pattern visualization

In [ ]:
# Visualize the missingness MATRIX: are nulls clustered together or random?
# A simple way without external libs: convert to bool and heatmap.

fig, ax = plt.subplots(figsize=(11, 5))
# Use only columns with at least one missing value.
miss_cols = emp.columns[emp.isna().any()].tolist()
sns.heatmap(emp[miss_cols].isna().T, cbar=False, ax=ax, cmap="rocket_r")
ax.set_title("Missingness pattern (yellow = present, dark = missing)")
ax.set_xlabel("Row index")
ax.set_ylabel("Column")
plt.tight_layout()
plt.show()


**Patterns to look for:**
- **Random scatter** → MCAR-like
- **Whole columns dark** → those columns are always missing (drop them)
- **Whole rows dark** → those rows are mostly missing (drop them)
- **Co-missing columns** → two columns missing together = they come from the same source/system


## 7.3 Diagnostic: does missingness correlate with another column?

In [ ]:
# For each potentially-missing column, check: is the missingness predictable from other columns?
# If YES -> likely MAR (or MNAR if predictability is through the missing value itself).
# If NO  -> consistent with MCAR.

emp["salary_is_missing"] = emp["salary_inr"].isna()

# Cramér's V of salary-missingness with each categorical, point-biserial with each continuous.
print("Cramér's V of 'salary_is_missing' with categoricals:")
for col in ["department_clean", "job_level", "remote", "left"]:
    v = cramers_v(emp["salary_is_missing"], emp[col])
    print(f"  {col:<20s}: {v:.3f}")

print("\nPearson corr of 'salary_is_missing' with continuous columns:")
for col in ["age", "tenure_yrs"]:
    r = emp[["salary_is_missing", col]].astype(float).corr().iloc[0, 1]
    print(f"  {col:<20s}: {r:+.3f}")


## 7.4 Imputation strategies

In [ ]:
# Strategy menu. Pick based on (a) pattern type, (b) downstream model, (c) computational budget.

# Work with float for imputation demo so we can fill with mean/median that aren't integers.
s = emp["salary_inr"].astype("Float64")

# 1. SIMPLE: drop rows. OK only if missingness is MCAR and < 5%.
simple_drop = emp.dropna(subset=["salary_inr"])
print(f"After dropping missing salary: {len(simple_drop)} rows (lost {len(emp) - len(simple_drop)})")

# 2. SIMPLE: mean. Distorts variance, removes any relationship the missing value had.
imp_mean = s.fillna(s.mean())

# 3. SIMPLE: median. Robust to outliers — preferred for skewed data like salary.
imp_median = s.fillna(s.median())

# 4. BY GROUP: median per department. Almost always better than global median.
imp_group = emp.assign(salary_f=s).groupby("department_clean", observed=True)["salary_f"].transform(
    lambda x: x.fillna(x.median())
)

# 5. FORWARD/BACKWARD FILL: only valid when rows have a meaningful order (time series).
# imp_ffill = s.ffill()  # not appropriate here

# 6. MODEL-BASED (KNN): predict missing salary from other features.
from sklearn.impute import KNNImputer
X_for_knn = emp[["age", "tenure_yrs", "satisfaction"]].astype(float).copy()
X_for_knn["salary_inr"] = s.astype(float)
knn = KNNImputer(n_neighbors=5)
imp_knn = pd.DataFrame(knn.fit_transform(X_for_knn), columns=X_for_knn.columns)["salary_inr"]

print("\nDistribution of imputed values vs original (where imputed):")
imputed_idx = emp["salary_inr"].isna()
print(f"  Mean impute    : {imp_mean[imputed_idx].mean():>12,.0f}")
print(f"  Median impute  : {imp_median[imputed_idx].mean():>12,.0f}")
print(f"  Group-median   : {imp_group[imputed_idx].mean():>12,.0f}")
print(f"  KNN impute     : {imp_knn[imputed_idx.values].mean():>12,.0f}")


## 7.5 Missingness as a feature

**The trick most candidates miss:** missingness itself is often predictive. Add a binary indicator `column_was_missing` before imputing. This way the model can learn "people who hide their salary tend to leave more often" without you having to make a decision about it.

In [ ]:
# Add missingness indicators and check their predictive power.
emp["salary_was_missing"] = emp["salary_inr"].isna().astype(int)
emp["satisfaction_was_missing"] = emp["satisfaction"].isna().astype(int)

# Are these indicators correlated with the target?
print("Mean leave rate by whether salary was missing:")
print(emp.groupby("salary_was_missing", observed=True)["left"].mean().round(3))

# Drop the helper column we added in 7.3.
emp = emp.drop(columns=["salary_is_missing"])


**Interview line:** "I always add a binary `*_was_missing` indicator before imputation. If the missingness is informative (MNAR), the model can use that signal. If it's pure noise, the indicator gets zero weight."


---
**Part 7 checklist:**
- [ ] MCAR vs MAR vs MNAR — definition and a real example of each?
- [ ] Why can't you tell MAR from MNAR using only the available data?
- [ ] Five imputation strategies in order of complexity?
- [ ] What's the "missingness as a feature" trick?
- [ ] What's the danger of imputing the mean? (distorts variance and downstream correlation)


---
# Part 8 — Aggregations and reshaping

The verbs interviewers literally test. `groupby`, `agg`, `transform`, `apply`, `pivot_table`, `melt`, `merge` — these are the eight that come up in every coding round.

## 8.1 `groupby` — split-apply-combine mental model

`groupby` is one of pandas' most powerful primitives. The model is:
1. **Split** the DataFrame into groups by the key(s).
2. **Apply** a function (aggregation, transformation, filter, or arbitrary) to each group.
3. **Combine** the results back into a DataFrame or Series.

`groupby` itself is **lazy** — it returns a `GroupBy` object that doesn't compute anything until you call an action.

In [ ]:
# Basic aggregation: mean salary per department.
print(emp.groupby("department_clean", observed=True)["salary_inr"].mean().round(0))

# 'observed=True' is important when grouping by category dtype — without it, you get
# rows for unobserved category combinations (which is rarely what you want and slows things down).


## 8.2 `agg` — the workhorse

In [ ]:
# Multiple aggregations at once.
agg_result = (
    emp.groupby("department_clean", observed=True)
    .agg(
        n_employees     = ("employee_id", "count"),
        n_left          = ("left", "sum"),
        leave_rate      = ("left", "mean"),
        median_salary   = ("salary_inr", "median"),
        mean_tenure     = ("tenure_yrs", "mean"),
        max_tenure      = ("tenure_yrs", "max"),
    )
    .round(3)
    .sort_values("leave_rate", ascending=False)
)
agg_result


**Named aggregations are the modern syntax** (pandas 0.25+). The dict-of-funcs syntax (`{"salary_inr": "mean"}`) still works, but named aggs are:
- More readable (you name the output column explicitly)
- Type-safe-ish (mistakes show up as clear errors)
- Easy to chain

Avoid: `df.groupby(...).agg(["mean", "max"])` — produces a multi-index column header that's annoying to work with downstream.

## 8.3 `agg` vs `transform` vs `apply` — three return shapes

In [ ]:
# Each does something different. Memorize the return shape.

# agg: returns ONE row per group.
agg_out = emp.groupby("department_clean", observed=True)["salary_inr"].mean()
print(f"agg     -> shape {agg_out.shape}  (one row per group)")

# transform: returns SAME shape as input. Broadcasts the group's aggregate back to every row.
# Most common use: subtract the group mean from each row (centering by group).
trans_out = emp.groupby("department_clean", observed=True)["salary_inr"].transform("mean")
print(f"transform -> shape {trans_out.shape}  (same shape as input)")

# apply: returns WHATEVER the function returns. Most flexible, slowest, most error-prone.
apply_out = emp.groupby("department_clean", observed=True)["salary_inr"].apply(lambda s: s.describe())
print(f"apply   -> shape {apply_out.shape}  (depends on what you return)")


In [ ]:
# Realistic transform use: salary relative to department median.
emp["salary_pct_of_dept_median"] = (
    emp["salary_inr"] / emp.groupby("department_clean", observed=True)["salary_inr"].transform("median")
)

# Show top 5 over-paid relative to their department.
emp.dropna(subset=["salary_pct_of_dept_median"]).nlargest(5, "salary_pct_of_dept_median")[
    ["employee_id", "department_clean", "salary_inr", "salary_pct_of_dept_median"]
]


**The trap:** beginners use `apply` for everything because it's the most flexible. But `apply` is the slowest (Python-level loop per group) and the most error-prone (you have to know the exact return type). Always reach for `agg` or `transform` first.

## 8.4 Multi-key groupby

In [ ]:
# Group by two keys.
multi = (
    emp.groupby(["department_clean", "job_level"], observed=True)
    .agg(n=("employee_id", "count"), leave_rate=("left", "mean"))
    .round(3)
)
multi.head(10)


In [ ]:
# Default index after groupby is a MultiIndex. Two ways to handle:

# Option 1: reset_index() -> flatten to a regular DataFrame.
flat = multi.reset_index()
print("After reset_index:")
print(flat.head())

# Option 2: as_index=False at groupby time -> never get a MultiIndex.
no_index = (
    emp.groupby(["department_clean", "job_level"], observed=True, as_index=False)
    .agg(n=("employee_id", "count"), leave_rate=("left", "mean"))
    .round(3)
)
print("\nWith as_index=False:")
print(no_index.head())


**Style preference:** I default to `as_index=False` because chained operations (e.g., merging the groupby result back with the original frame) are cleaner without a MultiIndex.

## 8.5 `pivot_table` and `crosstab`

In [ ]:
# pivot_table: pretty much the same as agg, but reshaped wide.
# Rows = department, Columns = job_level, Values = leave rate.

pt = emp.pivot_table(
    index="department_clean",
    columns="job_level",
    values="left",
    aggfunc="mean",
    observed=True,
)
print("Leave rate by (department, job_level):")
print(pt.round(3))


In [ ]:
# Visualize as a heatmap.
fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(pt, annot=True, fmt=".2f", cmap="Reds", ax=ax,
            cbar_kws={"label": "Leave rate"})
ax.set_title("Leave rate by department and job level")
plt.tight_layout()
plt.show()


In [ ]:
# crosstab: similar but a higher-level API for COUNTS of two (or more) categorical variables.
# normalize='index' / 'columns' / 'all' to get row, column, or grand-total proportions.

print("crosstab (counts):")
print(pd.crosstab(emp["department_clean"], emp["job_level"]))

print("\ncrosstab (row-normalized):")
print(pd.crosstab(emp["department_clean"], emp["job_level"], normalize="index").round(2))


**When to use `pivot_table` vs `crosstab`:** they're almost equivalent. `pivot_table` is more flexible (any aggfunc), `crosstab` is shorthand for the count-of-pairs case. Use `crosstab` when you only need contingency tables.

## 8.6 `melt` and `stack` / `unstack` — wide ↔ long

In [ ]:
# Often, data comes in WIDE format (one column per metric) but you want LONG (one row per metric).
# melt() turns wide into long.

# Build a wide example: dept-level metrics.
wide = (
    emp.groupby("department_clean", observed=True)
    .agg(
        head_count=("employee_id", "count"),
        leave_rate=("left", "mean"),
        median_salary=("salary_inr", "median"),
    )
    .reset_index()
)
print("WIDE:")
print(wide)

# Melt to long.
long = wide.melt(
    id_vars="department_clean",
    value_vars=["head_count", "leave_rate", "median_salary"],
    var_name="metric",
    value_name="value",
)
print("\nLONG (after melt):")
print(long)


**Which to use when?**
- **Wide** is ergonomic for humans reading tables and for joining.
- **Long** ("tidy data") is required by seaborn for `hue=metric` plotting, and by many model pipelines.

Knowing the difference and being able to translate between them on demand is a basic-skill check.

## 8.7 `merge` — and the duplicate-key trap

In [ ]:
# Build a small "departments" lookup with extra info.
dept_info = pd.DataFrame({
    "department_clean": ["Engineering", "Sales", "HR", "Marketing", "Finance", "Operations"],
    "cost_center": ["CC-100", "CC-200", "CC-300", "CC-400", "CC-500", "CC-600"],
    "vp_name":     ["Alice", "Bob", "Carol", "Dave", "Eve", "Frank"],
})

# Inner join: only rows where the key is in both.
merged = emp.merge(dept_info, on="department_clean", how="left", validate="many_to_one")
print(f"After left join: {merged.shape}  (original was {emp.shape})")
print(merged[["employee_id", "department_clean", "cost_center", "vp_name"]].head())


**The validate= parameter is gold.** It enforces one of:
- `"one_to_one"` — both sides have unique keys
- `"one_to_many"` — left side has unique keys
- `"many_to_one"` — right side has unique keys (your lookup table)
- `"many_to_many"` — both sides may have duplicates

If the actual data violates the relationship, **`merge` raises** instead of silently producing a row-multiplication explosion. Always use `validate=` with lookup-table merges.

In [ ]:
# Demonstrate the row-multiplication trap.
bad_lookup = pd.DataFrame({
    "department_clean": ["Engineering", "Engineering", "Sales"],  # DUPLICATE key!
    "cost_center": ["CC-100", "CC-101", "CC-200"],
})

# Without validate: this silently creates duplicate rows for every Engineering employee.
try:
    bad_merge = emp.merge(bad_lookup, on="department_clean", how="left", validate="many_to_one")
    print("This should have failed but didn't.")
except pd.errors.MergeError as e:
    print(f"validate='many_to_one' correctly catches it:\n  {e}")

# Without validate, you'd silently double rows for the Engineering dept.
no_validate = emp.merge(bad_lookup, on="department_clean", how="left")
print(f"\nNo validate, original rows: {len(emp)}, merged rows: {len(no_validate)}  <- BLOWN UP")


**The four join types:**

| Join | Result |
|---|---|
| `how="inner"` | Only keys in both. |
| `how="left"` | All keys from left + matches from right + NaN where no match. |
| `how="right"` | Mirror of left. |
| `how="outer"` | Keys in either + NaN where no match. |

**Practical rule:** if you're enriching a fact table with a lookup, `how="left"` + `validate="many_to_one"` is the safe pattern.

## 8.8 Window functions: `rolling`, `expanding`, `shift`

In [ ]:
# Often you want "cumulative leave rate over time" — a rolling/expanding window.
# Build a tiny time-series-flavored frame from join_date.

ts = (
    emp.sort_values("join_date")
    .assign(month=lambda d: d["join_date"].dt.to_period("M").dt.to_timestamp())
    .groupby("month", as_index=False)
    .agg(n_joiners=("employee_id", "count"), n_left=("left", "sum"))
)
ts["cumulative_joiners"] = ts["n_joiners"].cumsum()
ts["cumulative_left"]    = ts["n_left"].cumsum()
ts["rolling_3mo_joiners"] = ts["n_joiners"].rolling(window=3, min_periods=1).mean()
ts["prev_month_joiners"]  = ts["n_joiners"].shift(1)  # lag

print(ts.head(10).round(2))


**Window-function vocab:**
- `cumsum`, `cummax`, `cummin`, `cumprod` — cumulative (expanding window).
- `rolling(window=K)` — fixed-width sliding window.
- `expanding()` — window grows from the start.
- `shift(periods=N)` — lag (positive) or lead (negative). Indispensable for time-series feature engineering.
- `diff()` — first difference; equivalent to `x - x.shift(1)`.


---
**Part 8 checklist:**
- [ ] `agg` vs `transform` vs `apply` — return shapes and when to use which?
- [ ] Named aggregations syntax — write one from memory?
- [ ] What does `validate="many_to_one"` do in `merge`?
- [ ] Wide vs long format — when does each matter?
- [ ] Five window functions and their use cases?


---
# Part 9 — Textual data EDA (deep)

Text is the dominant data type for AI engineering in 2026. Every RAG dataset, every chat log, every support ticket archive is fundamentally a text-EDA problem.

This section is **deep** — ~25 cells — because text has properties that numbers don't (vocabulary, syntax, encoding), and the questions you ask in an interview about a text dataset look entirely different from the ones you ask about a tabular one.

We'll build a small synthetic support-ticket corpus and walk through:
1. Length distributions (char and word)
2. Vocabulary statistics (type-token ratio, Zipf's law)
3. N-gram counts
4. Quality flags (URLs, emails, all-caps, code-mix)
5. Near-duplicate detection
6. **Embedding-based clustering preview** for free-text columns

## 9.1 Generating a realistic text corpus

We'll simulate ~600 support tickets across 4 latent topics (billing, login, performance, feature requests), with Hinglish noise sprinkled in, plus deliberate quality issues.

In [ ]:
def generate_tickets(n=600, seed=7):
    '''Generate a synthetic customer-support corpus.'''
    rng = np.random.default_rng(seed)

    # 4 latent topics with associated phrasings.
    topic_templates = {
        "billing": [
            "I was charged twice for my subscription this month, please refund.",
            "Why has my plan auto-renewed? I cancelled last week.",
            "Bill amount is wrong, I have only 5 users not 50.",
            "Card declined while paying, what to do?",
            "GST invoice not generated for my last 3 payments.",
            "Unable to download invoice for March 2026, kindly help asap.",
        ],
        "login": [
            "Cannot log in to my account, password reset email is not coming.",
            "2FA code not received on my phone for 2 days now.",
            "SSO with Google is broken since yesterday morning.",
            "Why is my account locked? I never violated any rules.",
            "Login page just keeps loading, no error message.",
            "Stuck at OTP screen, please help to login.",
        ],
        "performance": [
            "App is super slow today, taking minutes to load the dashboard.",
            "Reports are timing out for queries over 1 month range.",
            "Page crashes every time I click on the analytics tab.",
            "API rate limit hit very quickly, can you increase it?",
            "Dashboard takes 30+ seconds to render, used to be instant.",
            "Mobile app freezes on the home screen after login.",
        ],
        "feature": [
            "Please add dark mode, it's 2026 already!",
            "Can we get Slack integration for notifications?",
            "Request: export to PDF with charts included.",
            "Suggestion: ability to schedule reports weekly via email.",
            "Would love to see multi-language support, especially Hindi.",
            "Feature ask: bulk delete for archived items.",
        ],
    }

    # Hinglish/code-mix templates (to demo language detection).
    hinglish_templates = [
        "Sir mera payment fail ho gaya hai please dekho.",
        "Login nahi ho raha bhai, OTP bhi nahi aa raha.",
        "Bhaiya report download nahi ho rahi, error aa raha hai.",
        "Kal subah se app slow chal raha hai, kuch karo.",
        "Refund kab milega? 1 week ho gaya.",
    ]

    tickets, topics, langs = [], [], []
    for _ in range(n):
        topic = rng.choice(list(topic_templates.keys()), p=[0.35, 0.30, 0.20, 0.15])
        if rng.random() < 0.15:
            text = rng.choice(hinglish_templates)
            lang = "hinglish"
        else:
            text = rng.choice(topic_templates[topic])
            lang = "english"
        tickets.append(text)
        topics.append(topic)
        langs.append(lang)

    # Inject quality issues:
    # 1. 2% empty strings
    empty_idx = rng.choice(n, size=int(0.02 * n), replace=False)
    for i in empty_idx:
        tickets[i] = ""

    # 2. 3% all-caps SHOUTING tickets
    caps_idx = rng.choice(n, size=int(0.03 * n), replace=False)
    for i in caps_idx:
        tickets[i] = tickets[i].upper()

    # 3. 5% tickets contain URL
    url_idx = rng.choice(n, size=int(0.05 * n), replace=False)
    for i in url_idx:
        if tickets[i]:
            tickets[i] += " See screenshot: https://example.com/screenshot/abc123.png"

    # 4. 3% tickets contain emails or phone numbers (PII)
    pii_idx = rng.choice(n, size=int(0.03 * n), replace=False)
    for i in pii_idx:
        if tickets[i]:
            tickets[i] += f" Contact me at user{rng.integers(1,9999)}@example.com or +91-{rng.integers(7000000000, 9999999999)}."

    # 5. 4 exact duplicates and 4 near-duplicates
    dup_idx = rng.choice([i for i in range(n) if tickets[i]], size=4, replace=False)
    for i in dup_idx:
        # Append exact duplicate
        tickets.append(tickets[i])
        topics.append(topics[i])
        langs.append(langs[i])
    near_dup_idx = rng.choice([i for i in range(n) if tickets[i]], size=4, replace=False)
    for i in near_dup_idx:
        # Append slightly modified version (case + trailing punctuation)
        tickets.append(tickets[i].lower().rstrip(".") + "!")
        topics.append(topics[i])
        langs.append(langs[i])

    return pd.DataFrame({
        "ticket_id": [f"T{i:05d}" for i in range(len(tickets))],
        "text":      pd.array(tickets, dtype="string"),
        "true_topic": pd.array(topics, dtype="string"),
        "true_lang":  pd.array(langs, dtype="string"),
    })


tickets = generate_tickets(n=600)
print(f"Generated {len(tickets)} tickets.")
tickets.head()


## 9.2 Step 1 — Length statistics

The very first thing you check on a text column: how long are the documents? This determines tokenization strategy, context-window budget, batch sizes — everything downstream.

In [ ]:
# Character length and word count.
tickets["n_chars"] = tickets["text"].str.len().fillna(0).astype(int)
tickets["n_words"] = tickets["text"].fillna("").str.split().str.len().astype(int)

print("Character length stats:")
print(tickets["n_chars"].describe().round(1))

print("\nWord count stats:")
print(tickets["n_words"].describe().round(1))

print(f"\nEmpty rows: {(tickets['n_chars'] == 0).sum()}")
print(f"Very short (<5 words): {(tickets['n_words'] < 5).sum()}")
print(f"Long (>50 words): {(tickets['n_words'] > 50).sum()}")


In [ ]:
# Plot length distributions — char count and word count side by side.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.histplot(tickets.loc[tickets["n_chars"] > 0, "n_chars"], bins=30, ax=axes[0], color="steelblue")
axes[0].set_title("Character length distribution (non-empty)")
axes[0].set_xlabel("# characters")

sns.histplot(tickets.loc[tickets["n_words"] > 0, "n_words"], bins=30, ax=axes[1], color="darkgreen")
axes[1].set_title("Word count distribution (non-empty)")
axes[1].set_xlabel("# words")

plt.tight_layout()
plt.show()


**What to ask the interviewer:**
- "What's the max context window I should plan for? Are there a few outliers I should truncate?"
- "Are very short tickets (1-3 words) real, or system-generated stubs?"
- "Are empty tickets data-loss bugs or intentional 'no comment' placeholders?"


## 9.3 Step 2 — Vocabulary statistics

In [ ]:
# Build a corpus-level token list (simple whitespace tokenization for EDA purposes).
import re
from collections import Counter

def simple_tokenize(text):
    '''Lowercased word-character tokens. Good enough for EDA. NOT what you'd use for modeling.'''
    if text is pd.NA or pd.isna(text):
        return []
    return re.findall(r"[a-zA-Z]+", str(text).lower())


all_tokens = [t for text in tickets["text"] for t in simple_tokenize(text)]
vocab = Counter(all_tokens)

print(f"Total tokens (corpus size N):  {len(all_tokens):,}")
print(f"Unique tokens (vocab size V):  {len(vocab):,}")
print(f"Type-token ratio (V/N):        {len(vocab)/len(all_tokens):.3f}")
print(f"Hapax legomena (words appearing exactly once): {sum(1 for c in vocab.values() if c == 1):,}")
print(f"Hapax fraction of vocab:       {sum(1 for c in vocab.values() if c == 1)/len(vocab):.2%}")


**What these numbers mean:**
- **Type-token ratio** (V/N): the closer to 1, the more lexically diverse. Real text is usually 0.05–0.15. Very high TTR on a small corpus is normal; on a large corpus, it suggests very specialized vocabulary (e.g., medical) or lots of noise.
- **Hapax legomena**: words that appear exactly once. In real text, ~40-50% of vocab is hapax — Zipf's law in action. If your hapax fraction is unusually high, you have a noise problem (typos, IDs, hashes leaking in).


## 9.4 Step 3 — Zipf's law check

In [ ]:
# Zipf's law: in natural text, frequency rank * frequency is approximately constant.
# Equivalently: log(rank) vs log(freq) is approximately a straight line with slope ~ -1.

freqs = sorted(vocab.values(), reverse=True)
ranks = np.arange(1, len(freqs) + 1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.loglog(ranks, freqs, marker=".", linestyle="", color="steelblue", alpha=0.6)
# Reference line: slope -1 (Zipf's ideal).
ax.loglog(ranks, freqs[0] / ranks, color="red", linestyle="--", label="Zipf reference (slope=-1)")
ax.set_xlabel("Rank (log scale)")
ax.set_ylabel("Frequency (log scale)")
ax.set_title("Zipf plot: vocabulary frequency vs rank")
ax.legend()
plt.show()

# Show top 10 most frequent tokens.
print("Top 10 most frequent tokens:")
for word, count in vocab.most_common(10):
    print(f"  {word:<12s}: {count}")


**Why care about Zipf?** Real-language data follows it. A corpus that **doesn't** follow Zipf usually has:
- Machine-generated content (logs, IDs)
- Aggressive deduplication (kills the head of the distribution)
- Heavy preprocessing already applied

If you build a vocabulary for tokenization and the Zipf plot is *flat*, your tokenizer is treating noise as words.

## 9.5 Step 4 — N-gram counts

In [ ]:
# Bigrams (2-word sequences) — often more informative than unigrams for topic discovery.
def ngrams(tokens, n=2):
    return [" ".join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

all_bigrams = []
for text in tickets["text"]:
    toks = simple_tokenize(text)
    all_bigrams.extend(ngrams(toks, n=2))

bigram_counts = Counter(all_bigrams)
print("Top 15 bigrams:")
for bg, c in bigram_counts.most_common(15):
    print(f"  {bg:<28s}: {c}")


**Read it:** the top bigrams ("not coming", "login page", "rate limit") give you a *much* better sense of the corpus's topics than the unigrams did. For richer topic discovery, n=2 and n=3 are usually the sweet spot. n=4+ gets sparse fast.

**Practical tip:** in production, use `sklearn.feature_extraction.text.CountVectorizer(ngram_range=(1, 2))` — handles tokenization, vocabulary building, and sparse matrix output in one call.

## 9.6 Step 5 — Quality flags via regex

In [ ]:
# A function that produces a row of quality flags per document.
URL_RE   = re.compile(r"https?://\S+|www\.\S+")
EMAIL_RE = re.compile(r"[\w.+-]+@[\w.-]+\.\w+")
PHONE_RE = re.compile(r"(?:\+?\d{1,3}[-\s]?)?\d{10}")  # naive: 10-digit numbers
HINDI_RE = re.compile(r"[\u0900-\u097F]")  # Devanagari script (matches actual Hindi script, not Hinglish)

def quality_flags(text):
    if pd.isna(text) or text == "":
        return {"is_empty": True, "is_allcaps": False, "n_urls": 0,
                "n_emails": 0, "n_phones": 0, "has_devanagari": False, "non_ascii_ratio": 0.0}
    s = str(text)
    return {
        "is_empty":       False,
        "is_allcaps":     s.isupper() and any(c.isalpha() for c in s),
        "n_urls":         len(URL_RE.findall(s)),
        "n_emails":       len(EMAIL_RE.findall(s)),
        "n_phones":       len(PHONE_RE.findall(s)),
        "has_devanagari": bool(HINDI_RE.search(s)),
        "non_ascii_ratio":sum(1 for c in s if ord(c) > 127) / len(s),
    }


qf = tickets["text"].apply(quality_flags).apply(pd.Series)
tickets_with_flags = pd.concat([tickets, qf], axis=1)

# Summary.
print("Quality flag counts:")
print(qf.sum().to_string())
print(f"\nAll-caps tickets sample:")
sample_caps = tickets_with_flags.loc[tickets_with_flags["is_allcaps"], "text"].head(3)
for t in sample_caps:
    print(f"  - {t[:80]}")


**Why each flag matters:**

| Flag | Why you check it |
|---|---|
| `is_empty` | Rows that need dropping (or marking) |
| `is_allcaps` | Often = angry/spam/auto-generated. Affects downstream model behavior. |
| `n_urls`, `n_emails`, `n_phones` | **PII detection** — must be redacted before storing/training |
| `has_devanagari` | Real Hindi script content (not Hinglish — that's still ASCII) |
| `non_ascii_ratio` | High ratio = non-English content (or emoji-heavy social text) |

**Interview line for AI engineering specifically:** "Before feeding text into an embedding model or LLM, I always run a quality-flag pass. The big three are: empty strings (cause embedding errors), PII (legal liability), and language detection (because most embedding models are trained mostly on English and degrade on Indic languages — relevant for Indian datasets)."


## 9.7 Step 6 — Hinglish / code-mix detection (cheap heuristic)

In [ ]:
# Detecting Hinglish is genuinely hard. A cheap heuristic:
# Hinglish text is mostly ASCII (Latin alphabet), but uses Hindi words spelled in Roman.
# Check for tokens that match common Hindi-spelled-in-Roman markers.

HINGLISH_MARKERS = {
    "hai", "nahi", "kya", "kar", "bhai", "bhaiya", "sir", "mera", "tera",
    "kab", "kyun", "kyon", "milega", "raha", "rahi", "raho", "ho", "se",
    "ka", "ki", "ko", "me", "mein", "pe", "par", "tak", "abhi", "kal",
}

def hinglish_score(text):
    if pd.isna(text) or text == "":
        return 0.0
    toks = simple_tokenize(text)
    if not toks:
        return 0.0
    return sum(t in HINGLISH_MARKERS for t in toks) / len(toks)


tickets_with_flags["hinglish_score"] = tickets["text"].apply(hinglish_score)

# Compare estimated vs ground-truth language label.
fig, ax = plt.subplots(figsize=(9, 4))
sns.boxplot(data=tickets_with_flags, x="true_lang", y="hinglish_score", ax=ax)
ax.set_title("Hinglish-marker score by ground-truth language")
ax.set_ylabel("Fraction of tokens matching Hinglish markers")
plt.show()

print("Heuristic accuracy (threshold=0.1):")
predicted = (tickets_with_flags["hinglish_score"] > 0.1).map({True: "hinglish", False: "english"})
print((predicted == tickets_with_flags["true_lang"]).mean().round(3))


**For production:** use `fasttext` language ID (Facebook's pre-trained model handles ~176 languages including Indic ones) or `lingua-py`. The heuristic above is just for "I need a quick split right now."

**Why this matters specifically for Indian AI engineering:** Sarvam AI, Krutrim, and other Indian voice/LLM products all have to handle Hinglish and code-mixed text. If you say in an interview "I'd run a language-ID pass and route Hinglish text to a multilingual model rather than English-only," you immediately separate yourself from candidates who haven't thought about Indic NLP.

## 9.8 Step 7 — Near-duplicate detection

In [ ]:
# Step 7a: exact duplicates — trivial with pandas.
exact_dupes = tickets[tickets["text"].duplicated(keep=False) & (tickets["text"] != "")]
print(f"Exact-duplicate count: {len(exact_dupes)}")
print(exact_dupes.sort_values("text").head(6)[["ticket_id", "text"]])


In [ ]:
# Step 7b: near-duplicates after normalization.
# Normalize and re-check: lower, strip punctuation, collapse whitespace.

def normalize(text):
    if pd.isna(text) or text == "":
        return ""
    s = str(text).lower()
    s = re.sub(r"[^\w\s]", "", s)         # remove punctuation
    s = re.sub(r"\s+", " ", s).strip()    # collapse whitespace
    return s


tickets_with_flags["text_norm"] = tickets["text"].apply(normalize)
near_dupes = tickets_with_flags[
    tickets_with_flags["text_norm"].duplicated(keep=False) & (tickets_with_flags["text_norm"] != "")
]
print(f"Near-duplicate count after normalization: {len(near_dupes)}")


**For larger-scale near-dup detection:**
- **MinHash + LSH** (`datasketch` library) — finds Jaccard-similar documents in O(N) instead of O(N²).
- **SimHash** (Google) — similar idea but operates on hashes of feature vectors.
- **Embedding cosine similarity** — gold standard but expensive. We'll see this next.

**Why care about near-dupes?** They corrupt train/test splits (a near-dupe in train can leak its near-dupe partner into test, inflating metrics) and skew topic distributions.

## 9.9 Step 8 — Embedding-based clustering preview

The final and most powerful text-EDA move: **embed every document into a vector space, then cluster**. This surfaces topics you didn't know existed in the data and is the foundation of every modern RAG/labeling/curation workflow.

For pedagogical simplicity (and to avoid downloading a sentence-transformer model in this notebook), we'll use **TF-IDF + truncated SVD** as a stand-in for sentence embeddings. The workflow is identical to what you'd do with `sentence-transformers` — just swap the embedding step.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

# Filter to non-empty, non-Hinglish (so we have cleaner clusters for the demo).
text_df = tickets[(tickets["n_chars"] > 0)].copy().reset_index(drop=True)
print(f"Embedding {len(text_df)} documents")

# 1. EMBED: TF-IDF with 1-2 grams as our cheap embedding.
#    In production, swap this for: SentenceTransformer('BAAI/bge-small-en-v1.5').encode(text_df['text'])
vectorizer = TfidfVectorizer(
    max_features=2000,
    ngram_range=(1, 2),
    min_df=2,          # ignore terms in <2 documents
    max_df=0.95,       # ignore terms in >95% of documents (corpus-stopwords)
    stop_words="english",
)
X_sparse = vectorizer.fit_transform(text_df["text"].astype(str))
print(f"TF-IDF matrix shape: {X_sparse.shape}")

# 2. REDUCE: SVD to 50 dimensions (mimicking what embedding models output).
svd = TruncatedSVD(n_components=50, random_state=42)
X_dense = svd.fit_transform(X_sparse)
print(f"After SVD: {X_dense.shape}")
print(f"Explained variance: {svd.explained_variance_ratio_.sum():.2%}")


In [ ]:
# 3. CLUSTER: KMeans. Try a few K values and pick by silhouette.
candidate_ks = [2, 3, 4, 5, 6, 7, 8]
sil_scores = []
for k in candidate_ks:
    km = KMeans(n_clusters=k, random_state=42, n_init="auto")
    labels = km.fit_predict(X_dense)
    sil = silhouette_score(X_dense, labels)
    sil_scores.append(sil)
    print(f"  k={k}: silhouette={sil:.3f}")

# Plot the elbow / silhouette curve.
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(candidate_ks, sil_scores, marker="o")
ax.set_xlabel("Number of clusters (k)")
ax.set_ylabel("Silhouette score")
ax.set_title("KMeans silhouette by k (higher = better-separated clusters)")
ax.grid(True, alpha=0.3)
plt.show()


In [ ]:
# 4. PICK k AND INSPECT.
best_k = candidate_ks[int(np.argmax(sil_scores))]
print(f"Best k by silhouette: {best_k}")

km = KMeans(n_clusters=best_k, random_state=42, n_init="auto")
text_df["cluster"] = km.fit_predict(X_dense)

# Inspect each cluster — show top TF-IDF terms (a poor man's "topic label").
terms = vectorizer.get_feature_names_out()
centroid_terms_per_cluster = {}
for c in range(best_k):
    # Project centroids back to TF-IDF space to find top terms.
    centroid_in_svd = km.cluster_centers_[c]
    centroid_in_tfidf = svd.inverse_transform(centroid_in_svd.reshape(1, -1)).ravel()
    top_idx = np.argsort(centroid_in_tfidf)[-8:][::-1]
    top_words = [terms[i] for i in top_idx]
    centroid_terms_per_cluster[c] = top_words
    n_in_cluster = (text_df["cluster"] == c).sum()
    print(f"Cluster {c} ({n_in_cluster} docs): {', '.join(top_words)}")


In [ ]:
# 5. VALIDATE: how well do our discovered clusters match the ground-truth topics?
# (In a real interview dataset you wouldn't have ground truth — this is just a sanity check.)
ari = adjusted_rand_score(text_df["true_topic"], text_df["cluster"])
print(f"Adjusted Rand Index (discovered vs true topics): {ari:.3f}")
print("ARI = 0 -> random clustering. ARI = 1 -> perfect match.")

# Look at the contingency table.
print("\nCluster vs true topic:")
print(pd.crosstab(text_df["true_topic"], text_df["cluster"]))


In [ ]:
# 6. VISUALIZE: project the SVD embeddings to 2D for plotting.
from sklearn.decomposition import PCA

X_2d = PCA(n_components=2, random_state=42).fit_transform(X_dense)
text_df["x"] = X_2d[:, 0]
text_df["y"] = X_2d[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: colored by discovered cluster.
sns.scatterplot(data=text_df, x="x", y="y", hue="cluster", palette="tab10",
                ax=axes[0], alpha=0.7, s=30, legend="full")
axes[0].set_title(f"Discovered clusters (k={best_k})")

# Right: colored by ground-truth topic.
sns.scatterplot(data=text_df, x="x", y="y", hue="true_topic", palette="Set2",
                ax=axes[1], alpha=0.7, s=30, legend="full")
axes[1].set_title("Ground-truth topics")

plt.tight_layout()
plt.show()


## 9.10 The production upgrade path

The TF-IDF + SVD pipeline above gives you the **right mental model**. In production for AI engineering, swap each step for the modern equivalent:

| EDA pipeline step | Production replacement |
|---|---|
| `TfidfVectorizer` | `sentence-transformers` (BGE, E5, GTE) or OpenAI/Cohere/Voyage embeddings API |
| `TruncatedSVD` | Skip — embeddings are already dense |
| `KMeans` | HDBSCAN (handles variable cluster shapes + outliers) or topic models like BERTopic |
| `silhouette` for k | UMAP visualization + HDBSCAN's automatic cluster count |
| Manual centroid inspection | LLM-generated cluster labels (give an LLM 10 examples per cluster, ask for a topic name) |

The interview line: "For exploratory clustering on a new text corpus, my default 2026 pipeline is BGE-small embeddings → UMAP to 5D → HDBSCAN. For topic labels I prompt an LLM with 10 random docs per cluster. The TF-IDF + KMeans approach is what you do when you don't have GPU access or the corpus is too small to embed."


## 9.11 The text-EDA checklist for AI engineering interviews

When handed a text dataset, run through this in order:
1. **Length distributions** (char + word) — informs context window, batching
2. **Empty / near-empty rows** — drop or mark
3. **Vocab size + hapax fraction** — corpus health check
4. **Top n-grams (n=1, 2)** — quick topic hint
5. **Quality flags** — URLs, emails, phones (PII!), all-caps, non-ASCII
6. **Language ID** — especially for Indian/multilingual data
7. **Exact and near-dup detection** — corrupt splits if ignored
8. **Embedding cluster preview** — surface latent topics
9. **For LLM datasets specifically** — prompt/response length, refusal patterns, distribution of speaker roles

Questions to ask the interviewer about a text dataset:
- "Source: user-generated, synthetic, or scraped?"
- "Was any preprocessing (deduplication, PII redaction) already applied?"
- "Is it primarily one language? What about code-mixed examples?"
- "What's the max useful length — should I truncate at the embedding model's context, or summarize first?"
- "Is there a label hierarchy I should respect, or do I discover topics myself?"
- "Any known data poisoning concerns?"


---
**Part 9 checklist:**
- [ ] What's type-token ratio and what does a high or low value tell you?
- [ ] Zipf's law — what does the loglog plot look like for natural text?
- [ ] Five quality flags you'd run on a fresh text column?
- [ ] Why is Hinglish detection particularly relevant for Indian AI engineering?
- [ ] The 4-step embedding-cluster pipeline: embed, reduce, cluster, validate?
- [ ] Production upgrade for each step? (BGE / skip / HDBSCAN / LLM labels)


---
# Part 10 — End-to-end interview walkthrough

Now we put it all together. Imagine the interviewer drops a CSV in front of you. You have 10 minutes before they expect first observations. Here's the rehearsed routine.

We'll use a NEW synthetic dataset — a small e-commerce orders table — to demonstrate the routine cleanly. Read this section in one pass, then practice on your own datasets.

In [ ]:
def generate_orders(n=1500, seed=11):
    '''Synthetic e-commerce orders dataset for the walkthrough.'''
    rng = np.random.default_rng(seed)

    order_id = np.array([f"O{i:07d}" for i in range(n)])
    user_id  = np.array([f"U{rng.integers(1, 400):05d}" for _ in range(n)])  # repeat customers
    order_dt = pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 500, size=n), unit="D")
    category = rng.choice(["Electronics", "Apparel", "Home", "Books", "Beauty"],
                           size=n, p=[0.30, 0.30, 0.20, 0.10, 0.10])
    payment  = rng.choice(["UPI", "Card", "COD", "Wallet"], size=n, p=[0.55, 0.30, 0.10, 0.05])
    items    = rng.integers(1, 6, size=n)
    unit_price = np.round(rng.lognormal(mean=6.5, sigma=0.9, size=n), 0)
    total    = (items * unit_price).astype(int)
    discount_pct = rng.choice([0, 5, 10, 15, 20, 30], size=n, p=[0.4, 0.2, 0.2, 0.1, 0.05, 0.05])

    is_returned = rng.binomial(1, 0.08 + 0.04 * (category == "Apparel"), size=n).astype(bool)
    review_text = np.where(
        rng.random(n) < 0.4,
        rng.choice([
            "Great product, fast delivery!",
            "Product was damaged on arrival.",
            "Not as described, returning.",
            "Loved it, value for money.",
            "Average quality, expected better.",
        ], size=n),
        "",
    )

    df = pd.DataFrame({
        "order_id":   order_id,
        "user_id":    user_id,
        "order_dt":   order_dt,
        "category":   pd.array(category, dtype="string"),
        "payment":    pd.array(payment, dtype="string"),
        "items":      items,
        "unit_price": unit_price,
        "total":      total,
        "discount_pct": discount_pct,
        "is_returned": is_returned,
        "review_text": pd.array(review_text, dtype="string"),
    })

    # Inject realistic mess.
    # 1. Make total occasionally NULL (data plumbing bug).
    null_idx = rng.choice(n, size=int(0.04 * n), replace=False)
    df.loc[null_idx, "total"] = pd.NA

    # 2. Add 12 exact duplicates.
    dup_idx = rng.choice(n, size=12, replace=False)
    df = pd.concat([df, df.iloc[dup_idx]], ignore_index=True)

    df["total"] = df["total"].astype("Int64")
    df["items"] = df["items"].astype("Int64")
    return df


orders = generate_orders()
print(f"Got dataset: {orders.shape}")


## 10.1 Run the inspection script

In [ ]:
quick_eda(orders)


## 10.2 What you say out loud (the narration that wins the round)

After running the inspection above, here's the verbalization. **Practice saying this out loud.**

> "Okay, so we have ~1500 orders across 11 columns. Looking at the head — one row is one order, not one item. Let me confirm: do you want me to model at the order level or roll up to the customer level?"
>
> "On types — `order_id` is unique (a key), `user_id` repeats (so we have repeat customers, ~400 unique on 1500 orders, ~3.75 orders per user on average). `order_dt` covers about 500 days, roughly Jan 2025 through May 2026. `category` and `payment` are nominal with 5 and 4 levels respectively. `items` is a small integer, `unit_price` and `total` are continuous and look right-skewed. `is_returned` is boolean, target candidate. `review_text` is free text but very sparse — only ~40% filled."
>
> "Missingness: `total` is missing in 4% of rows. Given that `items * unit_price` should equal `total`, this is a derivation error, not real missingness. I'd recompute it. Also 12 exact duplicates — I want to confirm whether `order_id` should be unique."
>
> "A few questions before I go deeper: (1) what's the target? Is it return prediction, fraud, lifetime value? (2) Are returned orders excluded from revenue counts? (3) Why are there duplicate `order_id`s — is that a data extraction bug or expected?"

That narration takes ~45 seconds and demonstrates: feature classification, data quality awareness, leakage awareness, and the right questions. **This is what you're optimizing for.**

## 10.3 Compute a quick "things I'd worry about" list

In [ ]:
# Run the safety checks you'd narrate in section 10.2.

# 1. Are order_ids unique?
order_id_dupes = orders["order_id"].duplicated().sum()
print(f"Duplicate order_ids: {order_id_dupes}")

# 2. Does total == items * unit_price?
mask_can_check = orders["total"].notna()
recomputed = (orders.loc[mask_can_check, "items"] * orders.loc[mask_can_check, "unit_price"]).astype("Int64")
discrepancy = (orders.loc[mask_can_check, "total"] != recomputed).sum()
print(f"Rows where total != items * unit_price: {discrepancy}")

# 3. Are there future-dated orders (sanity check)?
future = (orders["order_dt"] > pd.Timestamp.now()).sum()
print(f"Future-dated orders: {future}")

# 4. Are there orders with zero or negative totals?
weird_totals = ((orders["total"] <= 0) & orders["total"].notna()).sum()
print(f"Orders with total <= 0: {weird_totals}")

# 5. How many users have only 1 order? Cold-start problem signal.
order_counts = orders.groupby("user_id", observed=True).size()
print(f"Single-order users: {(order_counts == 1).sum()} of {len(order_counts)}")
print(f"Median orders per user: {order_counts.median()}")


## 10.4 The "first chart" — always plot the target

In a real interview, after the textual narration, the single most useful plot is the **target distribution**.

In [ ]:
# Target: return rate overall, and per category.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: overall.
orders["is_returned"].value_counts(normalize=True).plot(
    kind="bar", ax=axes[0], color=["steelblue", "tomato"]
)
axes[0].set_title("Overall return rate")
axes[0].set_xticklabels(["Not returned", "Returned"], rotation=0)

# Right: by category.
cat_return = orders.groupby("category", observed=True)["is_returned"].mean().sort_values(ascending=False)
cat_return.plot(kind="bar", ax=axes[1], color="tomato")
axes[1].set_title("Return rate by category")
axes[1].set_xticklabels(cat_return.index, rotation=20)
axes[1].set_ylabel("Return rate")

plt.tight_layout()
plt.show()

print(f"Overall return rate: {orders['is_returned'].mean():.3f}")
print(f"Apparel return rate: {orders.loc[orders['category']=='Apparel', 'is_returned'].mean():.3f}")


**Narration:**
> "Overall return rate is ~10%. Apparel returns higher at ~12% — consistent with the industry pattern that size-fit issues dominate apparel returns. If we're modeling return prediction, I'd want a per-category model or strong category interactions, because the drivers will differ (apparel = fit, electronics = defect, books = wrong-item)."

That's a senior-level observation. **Always sanity-check your data against domain intuition you bring in.**

---
# Part 11 — Cheatsheet + final checklist

## 11.1 The single function to paste into any new notebook

In [ ]:
# Compact, copy-pasteable EDA function. Save this snippet for muscle memory.

def quick_eda_v2(df, max_unique_to_show=20):
    '''
    7-step EDA inspection.
    Usage in interview:
        quick_eda_v2(df)
    Then describe what you see + ask the 10 questions.
    '''
    print(f"\n{'='*72}")
    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} cols")
    print(f"{'='*72}\n")

    summary = pd.DataFrame({
        "dtype":      df.dtypes.astype(str),
        "n_missing":  df.isna().sum(),
        "miss_pct":   (df.isna().mean() * 100).round(1),
        "n_unique":   df.nunique(),
        "unique_pct": (df.nunique() / len(df) * 100).round(1),
    })
    # Add a "feature_type_guess" column using the decision tree.
    def guess(col):
        s = df[col]
        if s.nunique() / len(df) > 0.95: return "id_or_freetext"
        if pd.api.types.is_datetime64_any_dtype(s): return "datetime"
        if pd.api.types.is_bool_dtype(s): return "boolean"
        if pd.api.types.is_numeric_dtype(s) and s.nunique() <= 15: return "ordinal_or_discrete"
        if pd.api.types.is_numeric_dtype(s): return "continuous"
        return "categorical_or_text"
    summary["guess"] = [guess(c) for c in df.columns]
    print(summary)

    print(f"\n{'='*72}")
    print("describe (all):")
    print(f"{'='*72}")
    print(df.describe(include="all").T)

    print(f"\n{'='*72}")
    print("Sample (3 random rows):")
    print(f"{'='*72}")
    print(df.sample(min(3, len(df)), random_state=0))


quick_eda_v2(orders)


## 11.2 The 10 interviewer-questions (printable)

In [ ]:
QUESTIONS_TO_ASK = '''
The 10 questions to ask the interviewer about any dataset:

  1. What does one row represent? (unit of analysis)
  2. What is the prediction target, and how is it defined operationally?
  3. Is this a snapshot or a time-series panel?
  4. How was the data sampled? Is it representative?
  5. What's the time range, and are there structural breaks (re-orgs, COVID, product launches)?
  6. Are any columns derived from the target? (LEAKAGE risk)
  7. Are duplicates expected, accidental, or forbidden?
  8. What's the labeling process and the known error rate?
  9. Which columns are user-editable vs system-generated?
 10. What does 'missing' mean for each column?
     (not applicable / not yet recorded / user refused / data plumbing bug)
'''
print(QUESTIONS_TO_ASK)


## 11.3 The feature-type cheat sheet

In [ ]:
FEATURE_TYPE_PLAYBOOK = '''
                       check                   plot                question to ask
ID / key               uniqueness, dupes,      —                   should this be unique? what's the join key?
                       format consistency

Continuous numeric     range, skew, outliers,  hist + box          valid range? log-transform candidate?
                       units

Nominal categorical    cardinality, mode,      bar of counts       how many categories? rare ones meaningful?
                       rare cats, typos

Ordinal                respect order,          ordered bar         is spacing equal or just ordered?
                       per-rank distribution

Boolean                class balance           bar/pie             what does True mean operationally?

Datetime               range, gaps,            timeline            timezone? local or UTC? gaps expected?
                       seasonality, tz

Geo                    valid ranges,           map                 spatial resolution? user-reported?
                       missingness

Free text              length distribution,    length hist,        source: user/synthetic/scraped?
                       vocab, quality flags    n-gram bar          language ID needed?
'''
print(FEATURE_TYPE_PLAYBOOK)


## 11.4 The "tell-tale phrases" that flag a senior candidate

Memorize a few of these. Drop one into every EDA round.

| Situation | What to say |
|---|---|
| Big right skew | "I'd consider a log transform — mean is well above median, suggests a power-law tail." |
| Two features with corr ≈ 1.0 | "That's a leakage smell — one feature is likely a function of the other." |
| Class imbalance ~1:10 or worse | "I'd stratify the split, use class weights or focal loss, and report PR-AUC instead of ROC-AUC." |
| Lots of missing in one column | "I'd add a `_was_missing` indicator before imputing — missingness can itself be informative." |
| Categorical with 100+ levels | "High cardinality — I'd target-encode with out-of-fold, or hash, depending on the model." |
| Text column | "I'd run length stats, PII flags, language ID, and an embedding-cluster preview before deciding on a representation." |
| Time-based data | "I need to confirm the train/test split is time-based, not random — random split leaks future into past." |
| 1.0 correlation between Pearson and Spearman | "Pearson says linear, Spearman agrees so it's monotonic — but I should still plot it." |
| Boolean target with very few positives (<2%) | "I'd downsample negatives or use a rare-event-aware loss; metric-wise PR-AUC and recall at fixed precision matter more than accuracy." |

## 11.5 Final 20-point self-assessment

If you can confidently answer **15+** of these without looking, you're interview-ready for the EDA portion of any AI engineering or data scientist role.

- [ ] State the `axis=0` vs `axis=1` rule.
- [ ] Use `np.random.default_rng` instead of `np.random.seed`.
- [ ] Combine boolean masks with `&`, `|`, `~` (parenthesize each).
- [ ] Use `nansum` / `nanmean` to skip NaN at the array level.
- [ ] Difference between `int64` (no nulls) and `Int64` (nullable).
- [ ] When to use `category` dtype (low cardinality, repeating strings).
- [ ] What Copy-on-Write changed in pandas 3.0.
- [ ] `loc[a:b]` inclusive vs `iloc[a:b]` exclusive slicing.
- [ ] Four `read_csv` kwargs that show experience.
- [ ] Recite the 7-step inspection routine.
- [ ] Recite at least 7 of the 10 interviewer-questions.
- [ ] When to use IQR vs z-score vs modified z-score for outliers.
- [ ] When to use log vs Box-Cox vs Yeo-Johnson.
- [ ] Pearson vs Spearman vs Kendall — when each is right.
- [ ] What Cramér's V measures (categorical × categorical association, 0–1).
- [ ] MCAR vs MAR vs MNAR with one example each.
- [ ] The "missingness as a feature" trick.
- [ ] `agg` vs `transform` vs `apply` return shapes.
- [ ] What `validate="many_to_one"` does in `merge`.
- [ ] The 4-step embedding-cluster pipeline: embed, reduce, cluster, validate.


---

**End of masterclass.** The next masterclass in the data series will go beyond EDA into feature engineering, encoding decisions, train/test splitting strategies, and data leakage in pipelines. Tell me when you want it.
